
# Roadcast

**Tema:** Previsão de Ocorrência de Sinistros em Rodovias Federais Brasileiras com Técnicas de Machine Learning.

## Objetivo

Investigar a aplicação de técnicas de Machine Learning para prever a ocorrência diária de sinistros registrados pela Polícia Rodoviária Federal (PRF).

O problema foi estruturado como uma tarefa de **classificação binária**, com as seguintes definições:

- **Unidade espacial:** combinação entre Unidade Federativa (UF) e Rodovia Federal (BR);
- **Unidade temporal:** dia;
- **Unidade de análise:** UF × BR × dia;
- **Variável-alvo:** ocorrência de pelo menos um sinistro registrado na combinação UF × BR durante o dia previsto;
- **Fonte dos dados:** arquivos públicos da PRF, agrupados por ocorrência;
- **Período dos dados:** 2017 a 2025;
- **Período de modelagem:** 2018 a 2025.

## Metodologia

O notebook documenta as principais etapas do desenvolvimento experimental:

1. Consolidação e validação dos dados históricos da PRF;
2. Definição do universo de rodovias e construção da base diária;
3. Engenharia de atributos temporais e históricos;
4. Divisão cronológica dos dados em treinamento, validação e teste;
5. Desenvolvimento e avaliação dos modelos preditivos;
6. Análise do desempenho por rodovia e frequência de sinistros;
7. Experimento complementar com limiares adaptativos de classificação.

### Modelos avaliados

- Baseline histórico baseado nos últimos 30 dias;
- Regressão Logística;
- XGBoost.

### Divisão temporal

- **Treinamento:** 2018–2023;
- **Validação:** 2024;
- **Teste:** 2025.

### Métricas de avaliação

Average Precision (AP), ROC-AUC, precisão, revocação, F1-score e F2-score.

A escolha dos limiares de classificação prioriza o F2-score, atribuindo maior importância à revocação.

## Considerações metodológicas

As observações negativas representam dias sem sinistros registrados na base da PRF para determinada combinação UF × BR.

Os atributos históricos utilizam informações anteriores ao dia previsto, evitando vazamento temporal de informações.

O modelo não prevê a localização exata, a gravidade ou a quantidade de sinistros. Seu objetivo é estimar a probabilidade de ocorrência de pelo menos um registro em uma combinação UF × BR durante determinado dia.

## Fontes

- [Dados Abertos da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf)
- [Dicionário de acidentes da PRF](https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-acidentes)



## __1 - Construindo Base Geral__

In [6]:

from pathlib import Path
import pandas as pd

# 1. Identifique a raiz do projeto Roadcast
DIRETORIO_ATUAL = Path.cwd()

if DIRETORIO_ATUAL.name.lower() == "notebooks":
    RAIZ_PROJETO = DIRETORIO_ATUAL.parent
else:
    RAIZ_PROJETO = DIRETORIO_ATUAL

# 2. Configure as pastas de entrada e saída
PASTA_DADOS = RAIZ_PROJETO / "data"
ARQUIVO_SAIDA = PASTA_DADOS / "acidentes_prf_2017_2025.csv"

# 3. Defina os anos que serão consolidados
ANOS = range(2017, 2026)

bases = []
colunas_referencia = None

print(f"Raiz do projeto: {RAIZ_PROJETO}")
print(f"Pasta dos dados: {PASTA_DADOS}")

# 4. Leia e valide cada arquivo anual
for ano in ANOS:
    arquivo = PASTA_DADOS / f"datatran{ano}.csv"

    if not arquivo.exists():
        raise FileNotFoundError(
            f"Arquivo não encontrado: {arquivo}"
        )

    # Separador e codificação confirmados no teste
    df = pd.read_csv(
        arquivo,
        sep=";",
        dtype=str,
        encoding="cp1252"
    )

    # Remove espaços extras dos nomes das colunas
    df.columns = df.columns.str.strip()

    # Verifica nomes de colunas duplicados
    if df.columns.duplicated().any():
        duplicadas = df.columns[
            df.columns.duplicated()
        ].tolist()

        raise ValueError(
            f"Colunas duplicadas em {arquivo.name}: {duplicadas}"
        )

    # Compara os nomes das colunas entre os arquivos
    colunas_atuais = set(df.columns)

    if colunas_referencia is None:
        colunas_referencia = colunas_atuais
    elif colunas_atuais != colunas_referencia:
        faltantes = sorted(
            colunas_referencia - colunas_atuais
        )
        extras = sorted(
            colunas_atuais - colunas_referencia
        )

        raise ValueError(
            f"Schema diferente em {arquivo.name}.\n"
            f"Colunas faltantes: {faltantes}\n"
            f"Colunas extras: {extras}"
        )

    # Registra o ano de origem de cada registro
    df["ano_fonte"] = ano
    bases.append(df)

    print(
        f"{ano}: {len(df):,} registros lidos".replace(",", ".")
    )

# 5. Consolida os arquivos
df_consolidado = pd.concat(
    bases,
    ignore_index=True,
    sort=False
)

# 6. Coloca ano_fonte na primeira coluna
colunas = ["ano_fonte"] + [
    coluna
    for coluna in df_consolidado.columns
    if coluna != "ano_fonte"
]
df_consolidado = df_consolidado[colunas]

# 7. Valida o total de registros
total_esperado = sum(len(df) for df in bases)
total_obtido = len(df_consolidado)

assert total_obtido == total_esperado, (
    f"Total divergente: esperado {total_esperado}, "
    f"obtido {total_obtido}"
)

print("\n--- RESUMO DA CONSOLIDAÇÃO ---")
print(
    f"Total de registros: {total_obtido:,}".replace(",", ".")
)
print(f"Total de colunas: {len(df_consolidado.columns)}")

print("\nRegistros por ano:")
print(
    df_consolidado["ano_fonte"]
    .value_counts()
    .sort_index()
)

# 8. Salva o CSV consolidado
df_consolidado.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig"
)

print(f"\nArquivo salvo em: {ARQUIVO_SAIDA.resolve()}")

print("\nPrévia do resultado:")
display(df_consolidado.head())


Raiz do projeto: c:\Users\gabri\OneDrive\person projects\TCC\Roadcast
Pasta dos dados: c:\Users\gabri\OneDrive\person projects\TCC\Roadcast\data
2017: 89.567 registros lidos
2018: 69.333 registros lidos
2019: 67.558 registros lidos
2020: 63.585 registros lidos
2021: 64.567 registros lidos
2022: 64.606 registros lidos
2023: 67.766 registros lidos
2024: 73.202 registros lidos
2025: 72.529 registros lidos

--- RESUMO DA CONSOLIDAÇÃO ---
Total de registros: 632.713
Total de colunas: 31

Registros por ano:
ano_fonte
2017    89567
2018    69333
2019    67558
2020    63585
2021    64567
2022    64606
2023    67766
2024    73202
2025    72529
Name: count, dtype: int64

Arquivo salvo em: C:\Users\gabri\OneDrive\person projects\TCC\Roadcast\data\acidentes_prf_2017_2025.csv

Prévia do resultado:


,ano_fonte,id,data_inversa,dia_semana,horario,uf,br,km,municipio,causa_acidente,...,feridos_graves,ilesos,ignorados,feridos,veiculos,latitude,longitude,regional,delegacia,uop
0,2017,17,2017-01-01,domingo,01:45:00,RS,116,"34,9",VACARIA,Defeito Mecânico no Veículo,...,0,2,0,4,2,"-28,5071196","-50,941176",SPRF-RS,DEL05-RS,UOP03-DEL05-RS
1,2017,20,2017-01-01,domingo,01:00:00,PR,376,636,TIJUCAS DO SUL,Velocidade Incompatível,...,0,1,0,0,2,"-25,754","-49,1266",SPRF-PR,DEL01-PR,DEL7/1-UOP08/PR
2,2017,69,2017-01-01,domingo,04:40:00,BA,101,65,ENTRE RIOS,Condutor Dormindo,...,1,2,0,2,2,"-11,9618","-38,0953",SPRF-BA,DEL01-BA,UOP04-DEL01-BA
3,2017,106,2017-01-01,domingo,06:30:00,PA,316,"72,5",CASTANHAL,Falta de Atenção à Condução,...,0,3,0,0,3,"-1,2899799","-47,83483207",SPRF-PA,DEL01-PA,DEL19/1-UOP02/PA
4,2017,109,2017-01-01,domingo,09:00:00,GO,20,"220,5",POSSE,Defeito na Via,...,1,0,1,3,2,"-14,14220931","-46,32258922",SPRF-DF,DEL02-DF,UOP03-DEL02-DF


## __2. Validações__

### __2.1 - Conferência dos registros por ano__

In [7]:

from pathlib import Path
import pandas as pd

# 1. Localizar o arquivo consolidado
raiz_projeto = Path.cwd()

if raiz_projeto.name.lower() == "notebooks":
    raiz_projeto = raiz_projeto.parent

caminho_arquivo = raiz_projeto / "data" / "acidentes_prf_2017_2025.csv"

# 2. Carregar os dados sem alterar os tipos originais
df = pd.read_csv(
    caminho_arquivo,
    sep=",",
    encoding="utf-8-sig",
    dtype=str
)

# 3. Quantidades esperadas nos arquivos originais
registros_esperados = {
    "2017": 89567,
    "2018": 69333,
    "2019": 67558,
    "2020": 63585,
    "2021": 64567,
    "2022": 64606,
    "2023": 67766,
    "2024": 73202,
    "2025": 72529
}

# 4. Contar registros por ano no arquivo consolidado
registros_encontrados = df["ano_fonte"].value_counts()

# 5. Comparar as quantidades
validacao = pd.DataFrame({
    "esperado": pd.Series(registros_esperados),
    "encontrado": registros_encontrados
})

validacao = validacao.fillna(0).astype(int)
validacao["diferenca"] = (
    validacao["encontrado"] - validacao["esperado"]
)

validacao["status"] = validacao["diferenca"].apply(
    lambda x: "OK" if x == 0 else "VERIFICAR"
)

# 6. Exibir os resultados
print("=== VALIDAÇÃO DA CONSOLIDAÇÃO ===")
display(validacao)

print(f"\nTotal esperado: {sum(registros_esperados.values()):,}")
print(f"Total encontrado: {len(df):,}")
print(f"Total de colunas: {df.shape[1]}")

# 7. Verificar integridade geral
if (
    (validacao["status"] == "OK").all()
    and len(df) == sum(registros_esperados.values())
    and df.shape[1] == 31
):
    print("\nVALIDAÇÃO APROVADA: quantidades e colunas conferem.")
else:
    print("\nATENÇÃO: foram encontradas divergências.")


=== VALIDAÇÃO DA CONSOLIDAÇÃO ===


,esperado,encontrado,diferenca,status
2017,89567,89567,0,OK
2018,69333,69333,0,OK
2019,67558,67558,0,OK
2020,63585,63585,0,OK
2021,64567,64567,0,OK
2022,64606,64606,0,OK
2023,67766,67766,0,OK
2024,73202,73202,0,OK
2025,72529,72529,0,OK



Total esperado: 632,713
Total encontrado: 632,713
Total de colunas: 31

VALIDAÇÃO APROVADA: quantidades e colunas conferem.


### __2.2 - Identificação de registros duplicados__

In [ ]:
# 1. Verificar linhas completamente idênticas
duplicatas_completas = df.duplicated(keep=False)

qtd_linhas_duplicadas = duplicatas_completas.sum()
qtd_duplicatas_excedentes = df.duplicated().sum()

print("=== DUPLICATAS COMPLETAS ===")
print(f"Linhas envolvidas em duplicações: {qtd_linhas_duplicadas:,}")
print(f"Linhas excedentes: {qtd_duplicatas_excedentes:,}")

# 2. Verificar IDs repetidos (desconsiderando IDs ausentes)
ids_validos = df["id"].notna()

ids_duplicados = df.loc[ids_validos, "id"].duplicated(
    keep=False
)

df_ids_duplicados = df.loc[ids_validos].loc[
    ids_duplicados
].copy()

print("\n=== IDs REPETIDOS ===")
print(f"Linhas com IDs repetidos: {len(df_ids_duplicados):,}")
print(f"IDs distintos repetidos: {df_ids_duplicados['id'].nunique():,}")
print(f"IDs ausentes: {df['id'].isna().sum():,}")

# 3. Verificar IDs repetidos dentro do mesmo ano
duplicatas_id_ano = df.duplicated(
    subset=["ano_fonte", "id"],
    keep=False
) & df["id"].notna()

print("\n=== IDs REPETIDOS NO MESMO ANO ===")
print(f"Linhas envolvidas: {duplicatas_id_ano.sum():,}")

# 4. Exibir exemplos, caso existam
if not df_ids_duplicados.empty:
    print("\n=== EXEMPLOS DE IDs REPETIDOS ===")

    display(
        df_ids_duplicados[
            ["ano_fonte", "id", "data_inversa", "uf", "br", "km"]
        ].sort_values(["id", "ano_fonte"]).head(20)
    )

# 5. Resumo de duplicatas completas por ano
print("\n=== DUPLICATAS COMPLETAS POR ANO ===")

resumo_duplicatas = (
    df.loc[duplicatas_completas]
    .groupby("ano_fonte")
    .size()
    .reindex(sorted(df["ano_fonte"].dropna().unique()), fill_value=0)
)

display(resumo_duplicatas.to_frame(name="linhas_duplicadas"))


=== DUPLICATAS COMPLETAS ===
Linhas envolvidas em duplicações: 0
Linhas excedentes: 0

=== IDs REPETIDOS ===
Linhas com IDs repetidos: 0
IDs distintos repetidos: 0
IDs ausentes: 0

=== IDs REPETIDOS NO MESMO ANO ===
Linhas envolvidas: 0

=== DUPLICATAS COMPLETAS POR ANO ===


,linhas_duplicadas
ano_fonte,
2017,0
2018,0
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0
2025,0


### __2.3 - Validação de Datas__

In [ ]:
# 1. Converter datas para datetime (sem alterar o df original)
datas = pd.to_datetime(
    df["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

# 2. Identificar problemas
datas_ausentes = df["data_inversa"].isna() | (
    df["data_inversa"].str.strip() == ""
)

datas_invalidas = datas.isna() & ~datas_ausentes

anos_incompativeis = (
    datas.notna()
    & (datas.dt.year.astype("Int64").astype("string") != df["ano_fonte"])
).fillna(False)

fora_periodo = (
    datas.notna()
    & ~datas.dt.year.between(2017, 2025)
)

# 3. Exibir resumo geral
print("=== VALIDAÇÃO DAS DATAS ===")

print(f"Datas ausentes: {datas_ausentes.sum():,}")
print(f"Datas inválidas: {datas_invalidas.sum():,}")
print(f"Datas fora do período 2017–2025: {fora_periodo.sum():,}")
print(f"Datas incompatíveis com ano_fonte: {anos_incompativeis.sum():,}")

print(f"\nData mais antiga: {datas.min()}")
print(f"Data mais recente: {datas.max()}")

# 4. Resumo por ano
resumo_datas = pd.DataFrame({
    "registros": df.groupby("ano_fonte").size(),
    "datas_ausentes": datas_ausentes.groupby(df["ano_fonte"]).sum(),
    "datas_invalidas": datas_invalidas.groupby(df["ano_fonte"]).sum(),
    "anos_incompativeis": anos_incompativeis.groupby(df["ano_fonte"]).sum()
})

print("\n=== RESUMO POR ANO ===")
display(resumo_datas)

# 5. Exibir exemplos de registros problemáticos
problemas_datas = (
    datas_ausentes
    | datas_invalidas
    | anos_incompativeis
    | fora_periodo
)

if problemas_datas.any():
    print("\n=== EXEMPLOS DE REGISTROS COM PROBLEMAS ===")

    display(
        df.loc[
            problemas_datas,
            ["id", "ano_fonte", "data_inversa", "uf", "br", "km"]
        ].head(20)
    )
else:
    print("\nVALIDAÇÃO APROVADA: nenhuma inconsistência nas datas.")


=== VALIDAÇÃO DAS DATAS ===
Datas ausentes: 0
Datas inválidas: 0
Datas fora do período 2017–2025: 0
Datas incompatíveis com ano_fonte: 0

Data mais antiga: 2017-01-01 00:00:00
Data mais recente: 2025-12-31 00:00:00

=== RESUMO POR ANO ===


,registros,datas_ausentes,datas_invalidas,anos_incompativeis
ano_fonte,,,,
2017,89567,0,0,0
2018,69333,0,0,0
2019,67558,0,0,0
2020,63585,0,0,0
2021,64567,0,0,0
2022,64606,0,0,0
2023,67766,0,0,0
2024,73202,0,0,0
2025,72529,0,0,0



VALIDAÇÃO APROVADA: nenhuma inconsistência nas datas.


### __2.4 - Validação da localização dos sinistros__

In [ ]:

colunas_localizacao = ["uf", "br", "km", "latitude", "longitude"]

# 1. Identificar valores ausentes ou vazios
print("=== VALORES AUSENTES ===")

ausentes = pd.DataFrame({
    "ausentes": [
        (df[col].isna() | df[col].str.strip().eq("")).sum()
        for col in colunas_localizacao
    ]
}, index=colunas_localizacao)

ausentes["percentual"] = (
    ausentes["ausentes"] / len(df) * 100
).round(4)

display(ausentes)

# 2. Converter campos numéricos apenas para análise
# A base original permanece inalterada

km_numerico = pd.to_numeric(
    df["km"].str.strip().str.replace(",", ".", regex=False),
    errors="coerce"
)

latitude_numerica = pd.to_numeric(
    df["latitude"].str.strip().str.replace(",", ".", regex=False),
    errors="coerce"
)

longitude_numerica = pd.to_numeric(
    df["longitude"].str.strip().str.replace(",", ".", regex=False),
    errors="coerce"
)

br_numerica = pd.to_numeric(
    df["br"].str.strip(),
    errors="coerce"
)

# 3. Identificar valores não convertíveis
print("\n=== VALORES NÃO CONVERTÍVEIS ===")

campos_numericos = {
    "br": br_numerica,
    "km": km_numerico,
    "latitude": latitude_numerica,
    "longitude": longitude_numerica
}

for coluna, valores in campos_numericos.items():
    originais = df[coluna].notna() & df[coluna].str.strip().ne("")
    invalidos = originais & valores.isna()

    print(f"{coluna}: {invalidos.sum():,}")

# 4. Verificar valores fora de limites básicos
print("\n=== VALORES SUSPEITOS ===")

print(f"KM negativo: {(km_numerico < 0).sum():,}")
print(f"Latitude fora de [-90, 90]: {(latitude_numerica.abs() > 90).sum():,}")
print(f"Longitude fora de [-180, 180]: {(longitude_numerica.abs() > 180).sum():,}")

# 5. Verificar distribuição das UFs
print("\n=== REGISTROS POR UF ===")

display(
    df["uf"]
    .value_counts(dropna=False)
    .to_frame(name="registros")
)

# 6. Verificar intervalo dos quilômetros
print("\n=== ESTATÍSTICAS DOS QUILÔMETROS ===")

display(km_numerico.describe())

# 7. Identificar registros sem localização rodoviária utilizável
localizacao_incompleta = (
    df["uf"].isna() | df["uf"].str.strip().eq("")
    | br_numerica.isna()
    | km_numerico.isna()
    | (km_numerico < 0)
)

print("\n=== LOCALIZAÇÃO RODOVIÁRIA INCOMPLETA ===")
print(f"Registros afetados: {localizacao_incompleta.sum():,}")
print(f"Percentual: {localizacao_incompleta.mean() * 100:.4f}%")

# 8. Exemplos de registros problemáticos
if localizacao_incompleta.any():
    print("\n=== EXEMPLOS ===")

    display(
        df.loc[
            localizacao_incompleta,
            ["id", "ano_fonte", "uf", "br", "km", "latitude", "longitude"]
        ].head(15)
    )


=== VALORES AUSENTES ===


,ausentes,percentual
uf,0,0.0
br,0,0.0
km,0,0.0
latitude,0,0.0
longitude,0,0.0



=== VALORES NÃO CONVERTÍVEIS ===
br: 0
km: 0
latitude: 0
longitude: 0

=== VALORES SUSPEITOS ===
KM negativo: 0
Latitude fora de [-90, 90]: 5
Longitude fora de [-180, 180]: 33

=== REGISTROS POR UF ===


,registros
uf,
MG,83427
SC,74671
PR,70631
RJ,47076
RS,44123
SP,41954
BA,33978
GO,30362
PE,26009



=== ESTATÍSTICAS DOS QUILÔMETROS ===


count    632713.000000
mean        260.298543
std         227.490190
min           0.000000
25%          76.900000
50%         194.400000
75%         409.000000
max        1470.000000
Name: km, dtype: float64


=== LOCALIZAÇÃO RODOVIÁRIA INCOMPLETA ===
Registros afetados: 0
Percentual: 0.0000%


### __2.5 - Investigação das coordenadas geográficas__

In [ ]:

# 1. Identificar coordenadas fora dos limites globais
latitude_invalida = ~latitude_numerica.between(-90, 90)
longitude_invalida = ~longitude_numerica.between(-180, 180)

# 2. Identificar registros com pelo menos uma coordenada inválida
coordenadas_invalidas = latitude_invalida | longitude_invalida

print("=== RESUMO DAS COORDENADAS ===")

print(f"Latitudes inválidas: {latitude_invalida.sum():,}")
print(f"Longitudes inválidas: {longitude_invalida.sum():,}")
print(f"Registros únicos afetados: {coordenadas_invalidas.sum():,}")

# 3. Distribuição por ano
print("\n=== REGISTROS AFETADOS POR ANO ===")

resumo_anos = (
    df.loc[coordenadas_invalidas]
    .groupby("ano_fonte")
    .size()
    .reindex(sorted(df["ano_fonte"].unique()), fill_value=0)
)

display(resumo_anos.to_frame(name="registros_invalidos"))

# 4. Exibir os registros problemáticos
print("\n=== REGISTROS COM COORDENADAS INVÁLIDAS ===")

colunas_exibicao = [
    "id", "ano_fonte", "uf", "br", "km",
    "latitude", "longitude"
]

display(
    df.loc[coordenadas_invalidas, colunas_exibicao]
    .sort_values(["ano_fonte", "uf"])
    .reset_index(drop=True)
)

# 5. Verificar coordenadas fora de uma faixa ampla do Brasil
# Faixa aproximada, usada apenas para diagnóstico
fora_faixa_brasil = (
    ~latitude_numerica.between(-34, 6)
    | ~longitude_numerica.between(-74, -28)
)

print("\n=== VERIFICAÇÃO GEOGRÁFICA PRELIMINAR ===")
print(
    "Registros fora da faixa aproximada do Brasil:",
    f"{fora_faixa_brasil.sum():,}"
)


=== RESUMO DAS COORDENADAS ===
Latitudes inválidas: 5
Longitudes inválidas: 33
Registros únicos afetados: 33

=== REGISTROS AFETADOS POR ANO ===


,registros_invalidos
ano_fonte,
2017,33
2018,0
2019,0
2020,0
2021,0
2022,0
2023,0
2024,0
2025,0



=== REGISTROS COM COORDENADAS INVÁLIDAS ===


,id,ano_fonte,uf,br,km,latitude,longitude
0,1030,2017,BA,101,"323,4","12,55456353","-411,328125"
1,448,2017,BA,407,111,-1033382874,-4016764289
2,604,2017,GO,153,"196,1","-14,49150709","-409,15287495"
3,807,2017,MG,381,873,"-22,41850506","-405,96461356"
4,174,2017,MG,381,"442,9",-19843262,-43861116
5,469,2017,MG,50,"147,3","-18,73210312","-408,21195602"
6,1039,2017,MG,116,275,"-17,83965209","-401,51307038"
7,1040,2017,MG,381,"936,1","-22,80858465","313,72114604"
8,30566,2017,MG,381,590,"-20,54570034","315,3493908"
9,1645,2017,MT,364,"315,7",158,553



=== VERIFICAÇÃO GEOGRÁFICA PRELIMINAR ===
Registros fora da faixa aproximada do Brasil: 53


### __2.6 - Verificação da consistência dos campos rodoviários__

In [ ]:

# 1. Lista das 27 unidades federativas
ufs_validas = {
    "AC", "AL", "AP", "AM", "BA", "CE", "DF",
    "ES", "GO", "MA", "MT", "MS", "MG",
    "PA", "PB", "PR", "PE", "PI", "RJ",
    "RN", "RS", "RO", "RR", "SC", "SP",
    "SE", "TO"
}

# 2. Verificar UFs desconhecidas
ufs_invalidas = ~df["uf"].isin(ufs_validas)

print("=== VALIDAÇÃO DAS UFs ===")
print(f"UFs distintas: {df['uf'].nunique()}")
print(f"Registros com UF desconhecida: {ufs_invalidas.sum():,}")

if ufs_invalidas.any():
    print(df.loc[ufs_invalidas, "uf"].value_counts().to_string())

# 3. Verificar números de rodovia
print("\n=== VALIDAÇÃO DAS BRs ===")

br_nao_inteira = br_numerica.notna() & (br_numerica % 1 != 0)
br_fora_faixa = br_numerica.notna() & ~br_numerica.between(1, 999)

print(f"BRs distintas: {br_numerica.nunique():,}")
print(f"BRs não inteiras: {br_nao_inteira.sum():,}")
print(f"BRs fora da faixa 1–999: {br_fora_faixa.sum():,}")

print("\nBRs mais frequentes:")
print(df["br"].value_counts().head(10).to_string())

# 4. Verificar quilômetros
print("\n=== VALIDAÇÃO DOS QUILÔMETROS ===")

print(f"KM igual a zero: {(km_numerico == 0).sum():,}")
print(f"KM acima de 1.000: {(km_numerico > 1000).sum():,}")
print(f"KM negativo: {(km_numerico < 0).sum():,}")

# 5. Identificar registros com BR ou UF suspeita
suspeitos = ufs_invalidas | br_nao_inteira | br_fora_faixa

print("\n=== REGISTROS COM UF OU BR SUSPEITA ===")
print(f"Total: {suspeitos.sum():,}")

if suspeitos.any():
    print(
        df.loc[
            suspeitos,
            ["id", "ano_fonte", "uf", "br", "km"]
        ].head(30).to_string(index=False)
    )

# 6. Resumo por ano
print("\n=== RESUMO POR ANO ===")

resumo = pd.DataFrame({
    "registros": df.groupby("ano_fonte").size(),
    "uf_invalida": ufs_invalidas.groupby(df["ano_fonte"]).sum(),
    "br_suspeita": (
        br_nao_inteira | br_fora_faixa
    ).groupby(df["ano_fonte"]).sum(),
    "km_zero": (km_numerico == 0).groupby(df["ano_fonte"]).sum(),
    "km_acima_1000": (km_numerico > 1000).groupby(df["ano_fonte"]).sum()
})

print(resumo.to_string())


=== VALIDAÇÃO DAS UFs ===
UFs distintas: 27
Registros com UF desconhecida: 0

=== VALIDAÇÃO DAS BRs ===
BRs distintas: 134
BRs não inteiras: 0
BRs fora da faixa 1–999: 1,407

BRs mais frequentes:
br
101    106465
116     95057
381     31086
40      30040
153     24685
163     20679
364     20551
277     18525
376     17097
262     16411

=== VALIDAÇÃO DOS QUILÔMETROS ===
KM igual a zero: 2,668
KM acima de 1.000: 1,825
KM negativo: 0

=== REGISTROS COM UF OU BR SUSPEITA ===
Total: 1,407
   id ano_fonte uf br km
 1506      2017 MG  0  0
 1721      2017 BA  0  0
 2758      2017 MG  0  0
16789      2017 MG  0  0
19408      2017 PB  0  0
24358      2017 SC  0  0
26044      2017 DF  0  0
27599      2017 TO  0  0
30583      2017 MG  0  0
31957      2017 MT  0  0
37658      2017 RJ  0  0
38230      2017 RJ  0  0
40649      2017 RO  0  0
43363      2017 MS  0  0
43611      2017 MG  0  0
47490      2017 RJ  0  0
48176      2017 RS  0  0
48674      2017 PR  0  0
51192      2017 PE  0  0
55326    

### __2.7 - Investigação dos valores de BR e KM__

In [ ]:
# 1. Identificar situações específicas
br_zero = br_numerica == 0
br_fora_faixa = ~br_numerica.between(1, 999)
km_zero = km_numerico == 0

print("=== VALORES DE BR FORA DA FAIXA ===")

print(
    df.loc[br_fora_faixa, "br"]
    .value_counts()
    .to_string()
)

# 2. Investigar combinações de BR e KM
print("\n=== COMBINAÇÕES DE BR E KM ===")

print(f"BR = 0 e KM = 0: {(br_zero & km_zero).sum():,}")
print(f"BR = 0 e KM > 0: {(br_zero & (km_numerico > 0)).sum():,}")
print(f"BR válida e KM = 0: {(~br_fora_faixa & km_zero).sum():,}")

# 3. Distribuição dos registros com BR = 0
print("\n=== BR = 0 POR ANO ===")

print(
    df.loc[br_zero, "ano_fonte"]
    .value_counts()
    .sort_index()
    .to_string()
)

# 4. Distribuição dos registros com BR = 0 por UF
print("\n=== BR = 0 POR UF ===")

print(
    df.loc[br_zero, "uf"]
    .value_counts()
    .to_string()
)

# 5. Exemplos com BR válida e KM = 0
print("\n=== EXEMPLOS: BR VÁLIDA E KM = 0 ===")

colunas = ["id", "ano_fonte", "uf", "br", "km"]

print(
    df.loc[
        ~br_fora_faixa & km_zero,
        colunas
    ].head(15).to_string(index=False)
)



=== VALORES DE BR FORA DA FAIXA ===
br
0    1407

=== COMBINAÇÕES DE BR E KM ===
BR = 0 e KM = 0: 1,407
BR = 0 e KM > 0: 0
BR válida e KM = 0: 1,261

=== BR = 0 POR ANO ===
ano_fonte
2017    146
2018    115
2019     98
2020    162
2021    171
2022    199
2023    160
2024    189
2025    167

=== BR = 0 POR UF ===
uf
RJ    154
RS    148
PR    127
SC     89
MG     89
RO     82
SP     70
PE     60
GO     56
MS     52
PA     47
BA     46
MT     46
DF     44
CE     42
PB     38
TO     36
PI     31
RN     26
ES     25
RR     20
AM     18
MA     15
AC     14
SE     11
AL     11
AP     10

=== EXEMPLOS: BR VÁLIDA E KM = 0 ===
   id ano_fonte uf  br km
 5816      2017 SP 153  0
15093      2017 PR 277  0
15482      2017 MG 365  0
18492      2017 RJ 493  0
19306      2017 RJ 493  0
19447      2017 MA 135  0
21948      2017 AP 210  0
30792      2017 BA 101  0
34075      2017 RS 468  0
34564      2017 PA 316  0
38250      2017 RJ 465  0
38793      2017 MA 135  0
50471      2017 PA 316  0
52334      

### __2.8 - Diagnóstico de valores ausentes__

In [ ]:
# 1. Marcadores textuais comuns de ausência
marcadores_ausencia = {
    "", "nan", "null", "none", "na", "n/a",
    "não informado", "nao informado",
    "ignorado", "ignorada", "sem informação",
    "sem informacao"
}

# 2. Identificar valores ausentes por coluna
resumo_ausentes = []

for coluna in df.columns:

    valores = df[coluna].astype("string").str.strip()

    ausentes_reais = valores.isna()

    marcadores = (
        valores.str.lower()
        .isin(marcadores_ausencia)
        .fillna(False)
    )

    total = (ausentes_reais | marcadores).sum()

    resumo_ausentes.append({
        "coluna": coluna,
        "ausentes_reais": int(ausentes_reais.sum()),
        "marcadores_textuais": int(marcadores.sum()),
        "total_suspeito": int(total),
        "percentual": round(total / len(df) * 100, 4)
    })

# 3. Construir resumo
resumo_ausentes = pd.DataFrame(resumo_ausentes)

resumo_ausentes = resumo_ausentes.sort_values(
    "total_suspeito",
    ascending=False
)

# 4. Exibir resultado completo em texto
print("=== DIAGNÓSTICO DE VALORES AUSENTES ===")

print(
    resumo_ausentes.to_string(index=False)
)

# 5. Colunas com algum valor suspeito
colunas_afetadas = resumo_ausentes[
    resumo_ausentes["total_suspeito"] > 0
]

print("\n=== RESUMO GERAL ===")

print(f"Total de colunas: {len(df.columns)}")
print(f"Colunas com valores suspeitos: {len(colunas_afetadas)}")
print(f"Colunas sem valores suspeitos: {len(df.columns) - len(colunas_afetadas)}")

# 6. Valores mais frequentes nas colunas afetadas
print("\n=== VALORES FREQUENTES NAS COLUNAS AFETADAS ===")

for coluna in colunas_afetadas["coluna"]:

    print(f"\nColuna: {coluna}")

    print(
        df[coluna]
        .value_counts(dropna=False)
        .head(10)
        .to_string()
    )


=== DIAGNÓSTICO DE VALORES AUSENTES ===
                coluna  ausentes_reais  marcadores_textuais  total_suspeito  percentual
condicao_metereologica               0                 8948            8948      1.4142
           sentido_via               0                 1407            1407      0.2224
                   uop             393                    0             393      0.0621
             delegacia             238                    0             238      0.0376
         tipo_acidente              41                    0              41      0.0065
              regional              26                    0              26      0.0041
classificacao_acidente              10                    0              10      0.0016
                    br               0                    0               0      0.0000
                    uf               0                    0               0      0.0000
               horario               0                    0               0     

### __2.9 - Verificação de padrões entre campos suspeitos__

In [15]:

# === VALIDAÇÃO 9: RELAÇÃO ENTRE CAMPOS SUSPEITOS ===

# 1. Criar indicadores sem modificar o DataFrame original
br_zero = br_numerica.eq(0)
km_zero = km_numerico.eq(0)

sentido_nao_informado = (
    df["sentido_via"]
    .str.strip()
    .str.lower()
    .eq("não informado")
)

# 2. Verificar sobreposição
print("=== RELAÇÃO ENTRE BR, KM E SENTIDO DA VIA ===")

print(f"BR = 0: {br_zero.sum():,}")
print(f"KM = 0: {km_zero.sum():,}")
print(f"Sentido não informado: {sentido_nao_informado.sum():,}")

print(
    "BR = 0 e sentido não informado:",
    f"{(br_zero & sentido_nao_informado).sum():,}"
)

print(
    "BR válida e sentido não informado:",
    f"{(~br_zero & sentido_nao_informado).sum():,}"
)

# 3. Verificar coordenadas dos registros com BR = 0
coordenadas_globais_validas = (
    latitude_numerica.between(-90, 90)
    & longitude_numerica.between(-180, 180)
)

coordenadas_faixa_brasil = (
    latitude_numerica.between(-34, 6)
    & longitude_numerica.between(-74, -28)
)

print("\n=== COORDENADAS DOS REGISTROS COM BR = 0 ===")

print(
    "Coordenadas globalmente válidas:",
    f"{(br_zero & coordenadas_globais_validas).sum():,}"
)

print(
    "Coordenadas dentro da faixa aproximada do Brasil:",
    f"{(br_zero & coordenadas_faixa_brasil).sum():,}"
)

# 4. Distribuição por ano
print("\n=== RESUMO POR ANO ===")

resumo = pd.DataFrame({
    "br_zero": br_zero.groupby(df["ano_fonte"]).sum(),
    "sentido_nao_informado": sentido_nao_informado.groupby(
        df["ano_fonte"]
    ).sum(),
    "ambos": (
        br_zero & sentido_nao_informado
    ).groupby(df["ano_fonte"]).sum()
})

print(resumo.to_string())

# 5. Exemplos com BR não identificada
print("\n=== EXEMPLOS DE REGISTROS COM BR = 0 ===")

colunas = [
    "id", "ano_fonte", "uf", "municipio",
    "br", "km", "sentido_via",
    "latitude", "longitude"
]

print(
    df.loc[br_zero, colunas]
    .head(15)
    .to_string(index=False)
)


=== RELAÇÃO ENTRE BR, KM E SENTIDO DA VIA ===
BR = 0: 1,407
KM = 0: 2,668
Sentido não informado: 1,407
BR = 0 e sentido não informado: 1,407
BR válida e sentido não informado: 0

=== COORDENADAS DOS REGISTROS COM BR = 0 ===
Coordenadas globalmente válidas: 1,406
Coordenadas dentro da faixa aproximada do Brasil: 1,406

=== RESUMO POR ANO ===
           br_zero  sentido_nao_informado  ambos
ano_fonte                                       
2017           146                    146    146
2018           115                    115    115
2019            98                     98     98
2020           162                    162    162
2021           171                    171    171
2022           199                    199    199
2023           160                    160    160
2024           189                    189    189
2025           167                    167    167

=== EXEMPLOS DE REGISTROS COM BR = 0 ===
   id ano_fonte uf                   municipio br km   sentido_via     latit

## __3. Padronização dos dados__

### __3.1 - Inspeção dos tipos e formatos__

In [ ]:
colunas_analisar = [
    "id", "data_inversa", "horario",
    "uf", "br", "km",
    "latitude", "longitude"
]

print("=== TIPOS ATUAIS ===")
print(df[colunas_analisar].dtypes.to_string())

print("\n=== EXEMPLOS DE VALORES ===")

for coluna in colunas_analisar:
    print(f"\n{coluna}:")
    print(df[coluna].dropna().head(5).to_list())

print("\n=== FORMATOS DE HORÁRIO ===")

horarios = df["horario"].astype("string").str.strip()

formato_horario_valido = horarios.str.fullmatch(
    r"(?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d"
).fillna(False)

print(f"Horários no formato HH:MM:SS: {formato_horario_valido.sum():,}")
print(f"Horários fora do formato: {(~formato_horario_valido).sum():,}")

if (~formato_horario_valido).any():
    print("\nExemplos de horários fora do formato:")
    print(
        df.loc[~formato_horario_valido, "horario"]
        .value_counts(dropna=False)
        .head(20)
        .to_string()
    )

print("\n=== FORMATO DOS CAMPOS DECIMAIS ===")

for coluna in ["km", "latitude", "longitude"]:
    valores = df[coluna].astype("string")

    print(f"\n{coluna}:")
    print(f"Com vírgula decimal: {valores.str.contains(',', regex=False).sum():,}")
    print(f"Com ponto decimal: {valores.str.contains('.', regex=False).sum():,}")


=== TIPOS ATUAIS ===
id              object
data_inversa    object
horario         object
uf              object
br              object
km              object
latitude        object
longitude       object

=== EXEMPLOS DE VALORES ===

id:
['17', '20', '69', '106', '109']

data_inversa:
['2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01', '2017-01-01']

horario:
['01:45:00', '01:00:00', '04:40:00', '06:30:00', '09:00:00']

uf:
['RS', 'PR', 'BA', 'PA', 'GO']

br:
['116', '376', '101', '316', '20']

km:
['34,9', '636', '65', '72,5', '220,5']

latitude:
['-28,5071196', '-25,754', '-11,9618', '-1,2899799', '-14,14220931']

longitude:
['-50,941176', '-49,1266', '-38,0953', '-47,83483207', '-46,32258922']

=== FORMATOS DE HORÁRIO ===
Horários no formato HH:MM:SS: 632,713
Horários fora do formato: 0

=== FORMATO DOS CAMPOS DECIMAIS ===

km:
Com vírgula decimal: 277,853
Com ponto decimal: 0

latitude:
Com vírgula decimal: 632,162
Com ponto decimal: 0

longitude:
Com vírgula decimal: 632,148

### __3.2 - Padronização dos tipos de dados__

In [ ]:
# 1. Criar cópia de trabalho
df_tratado = df.copy()

# 2. Converter identificadores para inteiros
for coluna in ["id", "br", "ano_fonte"]:
    df_tratado[coluna] = pd.to_numeric(
        df_tratado[coluna],
        errors="coerce"
    ).astype("Int64")

# 3. Converter data
df_tratado["data_inversa"] = pd.to_datetime(
    df_tratado["data_inversa"],
    format="%Y-%m-%d",
    errors="coerce"
)

# 4. Converter horário
df_tratado["horario"] = pd.to_datetime(
    df_tratado["horario"],
    format="%H:%M:%S",
    errors="coerce"
).dt.time

# 5. Converter campos decimais
for coluna in ["km", "latitude", "longitude"]:

    valores = (
        df_tratado[coluna]
        .astype("string")
        .str.strip()
        .str.replace(",", ".", regex=False)
    )

    df_tratado[coluna] = pd.to_numeric(
        valores,
        errors="coerce"
    ).astype("float64")

# 6. Padronizar UF como texto
df_tratado["uf"] = (
    df_tratado["uf"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# 7. Comparar valores ausentes antes e depois
colunas_convertidas = [
    "id", "br", "ano_fonte", "data_inversa",
    "horario", "km", "latitude", "longitude"
]

print("=== VALIDAÇÃO DAS CONVERSÕES ===")

for coluna in colunas_convertidas:
    antes = df[coluna].isna().sum()
    depois = df_tratado[coluna].isna().sum()

    print(
        f"{coluna}: "
        f"ausentes antes={antes:,} | "
        f"depois={depois:,} | "
        f"novos ausentes={depois - antes:,}"
    )

# 8. Verificar dimensões
print("\n=== DIMENSÕES ===")

print(f"Original: {df.shape}")
print(f"Tratado: {df_tratado.shape}")

# 9. Exibir tipos finais
print("\n=== TIPOS APÓS CONVERSÃO ===")

print(
    df_tratado[colunas_convertidas + ["uf"]]
    .dtypes
    .to_string()
)

# 10. Exibir exemplos
print("\n=== EXEMPLOS APÓS CONVERSÃO ===")

print(
    df_tratado[
        ["id", "data_inversa", "horario",
         "uf", "br", "km", "latitude", "longitude"]
    ].head(5).to_string(index=False)
)


=== VALIDAÇÃO DAS CONVERSÕES ===
id: ausentes antes=0 | depois=0 | novos ausentes=0
br: ausentes antes=0 | depois=0 | novos ausentes=0
ano_fonte: ausentes antes=0 | depois=0 | novos ausentes=0
data_inversa: ausentes antes=0 | depois=0 | novos ausentes=0
horario: ausentes antes=0 | depois=0 | novos ausentes=0
km: ausentes antes=0 | depois=0 | novos ausentes=0
latitude: ausentes antes=0 | depois=0 | novos ausentes=0
longitude: ausentes antes=0 | depois=0 | novos ausentes=0

=== DIMENSÕES ===
Original: (632713, 31)
Tratado: (632713, 31)

=== TIPOS APÓS CONVERSÃO ===
id                       Int64
br                       Int64
ano_fonte                Int64
data_inversa    datetime64[ns]
horario                 object
km                     float64
latitude               float64
longitude              float64
uf              string[python]

=== EXEMPLOS APÓS CONVERSÃO ===
 id data_inversa  horario uf  br    km   latitude  longitude
 17   2017-01-01 01:45:00 RS 116  34.9 -28.507120 -50.941

### __3.3 - Inspeção das variáveis categóricas__

In [18]:

# === CÉLULA 12: INSPEÇÃO DAS VARIÁVEIS CATEGÓRICAS ===

colunas_categoricas = [
    "dia_semana",
    "municipio",
    "causa_acidente",
    "tipo_acidente",
    "classificacao_acidente",
    "fase_dia",
    "sentido_via",
    "condicao_metereologica",
    "tipo_pista",
    "tracado_via",
    "uso_solo",
    "regional",
    "delegacia",
    "uop"
]

print("=== RESUMO DAS VARIÁVEIS CATEGÓRICAS ===")

resumo = []

for coluna in colunas_categoricas:
    resumo.append({
        "coluna": coluna,
        "categorias_distintas": df_tratado[coluna].nunique(),
        "valores_ausentes": df_tratado[coluna].isna().sum()
    })

resumo = pd.DataFrame(resumo)

print(resumo.to_string(index=False))

# 2. Inspecionar valores mais frequentes
print("\n=== VALORES MAIS FREQUENTES ===")

for coluna in colunas_categoricas:

    print(f"\n--- {coluna.upper()} ---")

    frequencias = df_tratado[coluna].value_counts(
        dropna=False
    )

    print(f"Total de categorias: {df_tratado[coluna].nunique()}")

    print(
        frequencias.head(15).to_string()
    )

    if len(frequencias) > 15:
        print(f"... outras {len(frequencias) - 15} categorias/valores")

# 3. Verificar espaços em branco nas extremidades
print("\n=== VALORES COM ESPAÇOS NAS EXTREMIDADES ===")

for coluna in colunas_categoricas:

    valores = df_tratado[coluna].astype("string")

    com_espacos = (
        valores.notna()
        & valores.ne(valores.str.strip())
    ).sum()

    if com_espacos > 0:
        print(f"{coluna}: {com_espacos:,}")

print("\nInspeção concluída.")


=== RESUMO DAS VARIÁVEIS CATEGÓRICAS ===
                coluna  categorias_distintas  valores_ausentes
            dia_semana                     7                 0
             municipio                  2156                 0
        causa_acidente                    91                 0
         tipo_acidente                    21                41
classificacao_acidente                     3                10
              fase_dia                     4                 0
           sentido_via                     3                 0
condicao_metereologica                    10                 0
            tipo_pista                     3                 0
           tracado_via                  1328                 0
              uso_solo                     2                 0
              regional                    29                26
             delegacia                   156               238
                   uop                   440               393

=== VALORES M

### __3.4 - Consistência do dia da semana__

In [ ]:
# 1. Mapeamento dos dias da semana
mapa_dias = {
    0: "segunda-feira",
    1: "terça-feira",
    2: "quarta-feira",
    3: "quinta-feira",
    4: "sexta-feira",
    5: "sábado",
    6: "domingo"
}

# 2. Calcular o dia da semana a partir da data
dia_calculado = (
    df_tratado["data_inversa"]
    .dt.dayofweek
    .map(mapa_dias)
)

# 3. Comparar com a informação original
dia_informado = (
    df_tratado["dia_semana"]
    .astype("string")
    .str.strip()
    .str.lower()
)

divergencias = dia_informado.ne(dia_calculado).fillna(True)

# 4. Resumo geral
print("=== CONSISTÊNCIA DO DIA DA SEMANA ===")

print(f"Total de registros: {len(df_tratado):,}")
print(f"Registros consistentes: {(~divergencias).sum():,}")
print(f"Registros divergentes: {divergencias.sum():,}")

print(
    "Percentual de divergência:",
    f"{divergencias.mean() * 100:.4f}%"
)

# 5. Resumo por ano
print("\n=== DIVERGÊNCIAS POR ANO ===")

resumo = (
    divergencias
    .groupby(df_tratado["ano_fonte"])
    .sum()
)

print(resumo.to_string())

# 6. Exibir exemplos de divergências
if divergencias.any():

    print("\n=== EXEMPLOS DE DIVERGÊNCIAS ===")

    exemplos = df_tratado.loc[
        divergencias,
        ["id", "ano_fonte", "data_inversa", "dia_semana"]
    ].copy()

    exemplos["dia_calculado"] = dia_calculado.loc[
        divergencias
    ]

    print(exemplos.head(20).to_string(index=False))

else:
    print("\nVALIDAÇÃO APROVADA: todos os dias da semana conferem.")


=== CONSISTÊNCIA DO DIA DA SEMANA ===
Total de registros: 632,713
Registros consistentes: 632,713
Registros divergentes: 0
Percentual de divergência: 0.0000%

=== DIVERGÊNCIAS POR ANO ===
ano_fonte
2017    0
2018    0
2019    0
2020    0
2021    0
2022    0
2023    0
2024    0
2025    0

VALIDAÇÃO APROVADA: todos os dias da semana conferem.


## __4 - Definição das Unidades UF x BR__

In [ ]:
# 1. Definir critérios mínimos de elegibilidade
uf_valida = df_tratado["uf"].isin(ufs_validas)

br_valida = df_tratado["br"].between(1, 999).fillna(False)

data_valida = df_tratado["data_inversa"].notna()

elegivel = uf_valida & br_valida & data_valida

# 2. Criar base de ocorrências elegíveis
df_uf_br = df_tratado.loc[elegivel].copy()

# 3. Criar identificador da unidade espacial
df_uf_br["id_rodovia"] = (
    df_uf_br["uf"].astype(str)
    + "_BR"
    + df_uf_br["br"].astype(str).str.zfill(3)
)

# 4. Contabilizar ocorrências por UF e BR
resumo_rodovias = (
    df_uf_br
    .groupby(["uf", "br", "id_rodovia"])
    .agg(
        total_sinistros=("id", "size"),
        primeira_ocorrencia=("data_inversa", "min"),
        ultima_ocorrencia=("data_inversa", "max")
    )
    .reset_index()
    .sort_values("total_sinistros", ascending=False)
)

# 5. Resumo geral
print("=" * 80)
print("IDENTIFICAÇÃO DAS UNIDADES UF × BR")
print("=" * 80)

print(f"\nRegistros originais: {len(df_tratado):,}")
print(f"Registros elegíveis: {len(df_uf_br):,}")
print(f"Registros não elegíveis: {(~elegivel).sum():,}")

print("\n--- MOTIVOS DE INELEGIBILIDADE ---")
print(f"UF inválida: {(~uf_valida).sum():,}")
print(f"BR inválida: {(~br_valida).sum():,}")
print(f"Data inválida: {(~data_valida).sum():,}")

print("\n--- UNIDADES ESPACIAIS ---")
print(f"UFs distintas: {df_uf_br['uf'].nunique()}")
print(f"BRs distintas: {df_uf_br['br'].nunique()}")
print(f"Pares UF × BR distintos: {len(resumo_rodovias):,}")

# 6. Distribuição das rodovias por UF
print("\n--- QUANTIDADE DE PARES UF × BR POR ESTADO ---")
print(
    resumo_rodovias
    .groupby("uf")
    .size()
    .sort_values(ascending=False)
    .to_string()
)

# 7. Distribuição de sinistros por rodovia
print("\n--- ESTATÍSTICAS DE SINISTROS POR PAR UF × BR ---")
print(
    resumo_rodovias["total_sinistros"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
    .to_string()
)

# 8. Rodovias com mais sinistros
print("\n--- 15 PARES UF × BR COM MAIS SINISTROS ---")
print(
    resumo_rodovias[
        ["id_rodovia", "total_sinistros",
         "primeira_ocorrencia", "ultima_ocorrencia"]
    ]
    .head(15)
    .to_string(index=False)
)

# 9. Pares com poucos registros
print("\n--- PARES COM POUCOS SINISTROS NO PERÍODO ---")

for limite in [1, 5, 10, 30, 100]:
    quantidade = (
        resumo_rodovias["total_sinistros"] <= limite
    ).sum()

    print(f"Até {limite:3d} sinistros: {quantidade:,} pares")


IDENTIFICAÇÃO DAS UNIDADES UF × BR

Registros originais: 632,713
Registros elegíveis: 631,306
Registros não elegíveis: 1,407

--- MOTIVOS DE INELEGIBILIDADE ---
UF inválida: 0
BR inválida: 1,407
Data inválida: 0

--- UNIDADES ESPACIAIS ---
UFs distintas: 27
BRs distintas: 133
Pares UF × BR distintos: 266

--- QUANTIDADE DE PARES UF × BR POR ESTADO ---
uf
BA    22
MG    21
RS    18
PR    15
GO    15
PE    14
PB    12
PI    12
SC    12
MS    11
PA    10
RJ    10
SP     9
RN     9
CE     9
ES     8
MA     8
AL     7
DF     7
RO     7
MT     6
TO     6
RR     6
AM     4
AC     3
SE     3
AP     2

--- ESTATÍSTICAS DE SINISTROS POR PAR UF × BR ---
count          266.0
mean     2373.330827
std      4583.693869
min              1.0
25%            120.0
50%            637.0
75%          2264.25
90%           7101.0
95%          10193.5
max          37261.0

--- 15 PARES UF × BR COM MAIS SINISTROS ---
id_rodovia  total_sinistros primeira_ocorrencia ultima_ocorrencia
  SC_BR101            37261 

### __4.2 - Cobertura temporal e frequência diária de sinistros__

In [ ]:
# 1. Agrupar sinistros por UF, BR e dia
df_ocorrencias_diarias = (
    df_uf_br
    .groupby(["uf", "br", "id_rodovia", "data_inversa"])
    .size()
    .reset_index(name="total_sinistros")
)

# 2. Identificar o ano de cada observação
df_ocorrencias_diarias["ano"] = (
    df_ocorrencias_diarias["data_inversa"].dt.year
)

# 3. Contabilizar pares com registros por ano
pares_por_ano = (
    df_ocorrencias_diarias
    .groupby("ano")["id_rodovia"]
    .nunique()
)

# 4. Cobertura temporal dos pares
cobertura_pares = (
    df_ocorrencias_diarias
    .groupby("id_rodovia")
    .agg(
        anos_com_sinistros=("ano", "nunique"),
        dias_com_sinistros=("data_inversa", "nunique"),
        total_sinistros=("total_sinistros", "sum")
    )
    .reset_index()
)

# 5. Dias positivos por ano
dias_positivos_ano = (
    df_ocorrencias_diarias
    .groupby("ano")
    .size()
)

print("=" * 80)
print("SEÇÃO 4.2 — COBERTURA TEMPORAL DOS PARES UF × BR")
print("=" * 80)

print("\n--- PARES COM SINISTROS POR ANO ---")
print(pares_por_ano.to_string())

print("\n--- DISTRIBUIÇÃO DE ANOS COM SINISTROS POR PAR ---")
print(
    cobertura_pares["anos_com_sinistros"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n--- DIAS COM SINISTROS POR ANO ---")
print(dias_positivos_ano.to_string())

print("\n--- ESTATÍSTICAS DE DIAS POSITIVOS POR PAR ---")
print(
    cobertura_pares["dias_com_sinistros"]
    .describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
    )
    .to_string()
)

print("\n--- PARES COM MAIOR FREQUÊNCIA DE DIAS POSITIVOS ---")
print(
    cobertura_pares
    .sort_values("dias_com_sinistros", ascending=False)
    .head(15)
    .to_string(index=False)
)

print("\n--- PARES COM MENOR FREQUÊNCIA DE DIAS POSITIVOS ---")
print(
    cobertura_pares
    .sort_values("dias_com_sinistros")
    .head(15)
    .to_string(index=False)
)


SEÇÃO 4.2 — COBERTURA TEMPORAL DOS PARES UF × BR

--- PARES COM SINISTROS POR ANO ---
ano
2017    230
2018    228
2019    228
2020    231
2021    230
2022    232
2023    236
2024    233
2025    231

--- DISTRIBUIÇÃO DE ANOS COM SINISTROS POR PAR ---
anos_com_sinistros
1     17
2     13
3      3
4      4
5      5
6      8
7      1
8      4
9    211

--- DIAS COM SINISTROS POR ANO ---
ano
2017    32162
2018    28650
2019    28500
2020    28033
2021    27988
2022    28086
2023    28745
2024    29992
2025    29442

--- ESTATÍSTICAS DE DIAS POSITIVOS POR PAR ---
count     266.000000
mean      983.451128
std      1037.632232
min         1.000000
25%       110.000000
50%       576.000000
75%      1580.750000
90%      2870.000000
95%      3110.500000
max      3286.000000

--- PARES COM MAIOR FREQUÊNCIA DE DIAS POSITIVOS ---
id_rodovia  anos_com_sinistros  dias_com_sinistros  total_sinistros
  SC_BR101                   9                3286            37261
  SP_BR116                   9      

### __4.3 - Diagnóstico da frequência de ocorrência__

In [35]:

# === SEÇÃO 4.3: DIAGNÓSTICO DA FREQUÊNCIA DIÁRIA ===

# Período completo da base
data_inicio = df_uf_br["data_inversa"].min()
data_fim = df_uf_br["data_inversa"].max()

total_dias = (data_fim - data_inicio).days + 1

# Pares com sinistros em todos os 9 anos
pares_estaveis = cobertura_pares.loc[
    cobertura_pares["anos_com_sinistros"] == 9
].copy()

# Comparar cenários
cenarios = {
    "Todos os pares": cobertura_pares,
    "Pares com sinistros nos 9 anos": pares_estaveis
}

print("=" * 80)
print("SEÇÃO 4.3 — FREQUÊNCIA DIÁRIA DE SINISTROS")
print("=" * 80)

print(f"\nPeríodo: {data_inicio.date()} a {data_fim.date()}")
print(f"Total de dias: {total_dias:,}")

for nome, base in cenarios.items():

    total_pares = len(base)
    observacoes_possiveis = total_pares * total_dias
    dias_positivos = int(base["dias_com_sinistros"].sum())

    taxa_positivos = (
        100 * dias_positivos / observacoes_possiveis
    )

    print(f"\n--- {nome.upper()} ---")
    print(f"Pares UF × BR: {total_pares:,}")
    print(f"Observações possíveis: {observacoes_possiveis:,}")
    print(f"Dias positivos: {dias_positivos:,}")
    print(f"Taxa estimada de positivos: {taxa_positivos:.2f}%")
    print(f"Taxa complementar: {100 - taxa_positivos:.2f}%")

# Frequência positiva por par
diagnostico_frequencia = cobertura_pares.copy()

diagnostico_frequencia["taxa_positivos_pct"] = (
    100 * diagnostico_frequencia["dias_com_sinistros"]
    / total_dias
)

print("\n--- DISTRIBUIÇÃO DA FREQUÊNCIA POR PAR ---")
print(
    diagnostico_frequencia["taxa_positivos_pct"]
    .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90])
    .to_string()
)

print("\n--- QUANTIDADE DE PARES POR FAIXA DE FREQUÊNCIA ---")

faixas = pd.cut(
    diagnostico_frequencia["taxa_positivos_pct"],
    bins=[-0.01, 1, 5, 20, 50, 80, 100],
    labels=[
        "Até 1%",
        ">1% a 5%",
        ">5% a 20%",
        ">20% a 50%",
        ">50% a 80%",
        ">80% a 100%"
    ]
)

print(
    faixas.value_counts()
    .sort_index()
    .to_string()
)

print("\nOBSERVAÇÃO:")
print(
    "As taxas assumem, apenas para diagnóstico, "
    "que todos os pares existiam durante todo o período."
)


SEÇÃO 4.3 — FREQUÊNCIA DIÁRIA DE SINISTROS

Período: 2017-01-01 a 2025-12-31
Total de dias: 3,287

--- TODOS OS PARES ---
Pares UF × BR: 266
Observações possíveis: 874,342
Dias positivos: 261,598
Taxa estimada de positivos: 29.92%
Taxa complementar: 70.08%

--- PARES COM SINISTROS NOS 9 ANOS ---
Pares UF × BR: 211
Observações possíveis: 693,557
Dias positivos: 260,029
Taxa estimada de positivos: 37.49%
Taxa complementar: 62.51%

--- DISTRIBUIÇÃO DA FREQUÊNCIA POR PAR ---
count    266.000000
mean      29.919414
std       31.567759
min        0.030423
10%        0.167326
25%        3.346517
50%       17.523578
75%       48.090964
90%       87.313660
max       99.969577

--- QUANTIDADE DE PARES POR FAIXA DE FREQUÊNCIA ---
taxa_positivos_pct
Até 1%         50
>1% a 5%       29
>5% a 20%      59
>20% a 50%     64
>50% a 80%     29
>80% a 100%    35

OBSERVAÇÃO:
As taxas assumem, apenas para diagnóstico, que todos os pares existiam durante todo o período.


### __4.4 - Primeira ocorrência e estabilidade dos pares__

In [ ]:
# Identificar primeira e última ocorrência por par
historico_pares = (
    df_uf_br
    .groupby("id_rodovia")
    .agg(
        primeira_ocorrencia=("data_inversa", "min"),
        ultima_ocorrencia=("data_inversa", "max"),
        total_sinistros=("id", "size")
    )
    .reset_index()
)

historico_pares["primeiro_ano"] = (
    historico_pares["primeira_ocorrencia"].dt.year
)

historico_pares["ultimo_ano"] = (
    historico_pares["ultima_ocorrencia"].dt.year
)

# Pares presentes no primeiro ano
pares_2017 = set(
    historico_pares.loc[
        historico_pares["primeiro_ano"] == 2017,
        "id_rodovia"
    ]
)

# Pares que apareceram posteriormente
pares_posteriores = historico_pares.loc[
    historico_pares["primeiro_ano"] > 2017
].copy()

# Distribuição da primeira ocorrência
print("=" * 80)
print("SEÇÃO 4.4 — HISTÓRICO TEMPORAL DOS PARES UF × BR")
print("=" * 80)

print("\n--- PRIMEIRO ANO COM SINISTRO REGISTRADO ---")
print(
    historico_pares["primeiro_ano"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n--- ÚLTIMO ANO COM SINISTRO REGISTRADO ---")
print(
    historico_pares["ultimo_ano"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n--- UNIVERSO INICIAL DE 2017 ---")
print(f"Pares identificados em 2017: {len(pares_2017):,}")
print(f"Pares identificados após 2017: {len(pares_posteriores):,}")

# Verificar quantos sinistros posteriores ocorreram em pares ainda não observados em 2017
sinistros_pares_novos = df_uf_br.loc[
    (df_uf_br["data_inversa"].dt.year > 2017)
    & (~df_uf_br["id_rodovia"].isin(pares_2017))
]

total_sinistros_posteriores = (
    df_uf_br["data_inversa"].dt.year > 2017
).sum()

print("\n--- IMPACTO DE UTILIZAR APENAS OS PARES DE 2017 ---")
print(
    "Sinistros de 2018–2025 em pares ausentes em 2017: "
    f"{len(sinistros_pares_novos):,}"
)

print(
    "Percentual dos sinistros de 2018–2025: "
    f"{100 * len(sinistros_pares_novos) / total_sinistros_posteriores:.2f}%"
)

print("\n--- PARES IDENTIFICADOS APÓS 2017 ---")
print(
    pares_posteriores[
        ["id_rodovia", "primeiro_ano", "ultimo_ano", "total_sinistros"]
    ]
    .sort_values(
        ["total_sinistros", "id_rodovia"],
        ascending=[False, True]
    )
    .head(20)
    .to_string(index=False)
)


SEÇÃO 4.4 — HISTÓRICO TEMPORAL DOS PARES UF × BR

--- PRIMEIRO ANO COM SINISTRO REGISTRADO ---
primeiro_ano
2017    230
2018      9
2019      8
2020      5
2021      1
2022      4
2023      3
2024      5
2025      1

--- ÚLTIMO ANO COM SINISTRO REGISTRADO ---
ultimo_ano
2017      2
2018      4
2019      7
2020      3
2021      1
2022      3
2023      6
2024      9
2025    231

--- UNIVERSO INICIAL DE 2017 ---
Pares identificados em 2017: 230
Pares identificados após 2017: 36

--- IMPACTO DE UTILIZAR APENAS OS PARES DE 2017 ---
Sinistros de 2018–2025 em pares ausentes em 2017: 977
Percentual dos sinistros de 2018–2025: 0.18%

--- PARES IDENTIFICADOS APÓS 2017 ---
id_rodovia  primeiro_ano  ultimo_ano  total_sinistros
  MS_BR376          2020        2025              410
  TO_BR010          2021        2025              352
  MS_BR487          2020        2025               45
  PI_BR235          2018        2025               35
  SC_BR285          2022        2025               14
  SP_

### __4.5 - Construção da base diária e variável-alvo__

In [ ]:
# 1. Definir universo de rodovias com base em 2017
universo_rodovias = (
    df_uf_br.loc[
        df_uf_br["data_inversa"].dt.year == 2017,
        ["uf", "br", "id_rodovia"]
    ]
    .drop_duplicates()
    .sort_values(["uf", "br"])
    .reset_index(drop=True)
)

# 2. Construir calendário diário de 2018 a 2025
calendario = pd.DataFrame({
    "data_inversa": pd.date_range(
        start="2018-01-01",
        end="2025-12-31",
        freq="D"
    )
})

# 3. Criar todas as combinações UF × BR × Dia
df_base_diaria = universo_rodovias.merge(
    calendario,
    how="cross"
)

# 4. Contar sinistros efetivamente registrados por par e dia
contagem_diaria = (
    df_ocorrencias_diarias.loc[
        df_ocorrencias_diarias["data_inversa"].between(
            "2018-01-01", "2025-12-31"
        ),
        ["id_rodovia", "data_inversa", "total_sinistros"]
    ]
)

# 5. Incorporar as contagens ao calendário
df_base_diaria = df_base_diaria.merge(
    contagem_diaria,
    on=["id_rodovia", "data_inversa"],
    how="left",
    validate="one_to_one"
)

df_base_diaria["total_sinistros"] = (
    df_base_diaria["total_sinistros"]
    .fillna(0)
    .astype(int)
)

# 6. Criar variável-alvo binária
df_base_diaria["ocorreu_sinistro"] = (
    df_base_diaria["total_sinistros"] > 0
).astype(int)

# 7. Ordenar registros
df_base_diaria = (
    df_base_diaria
    .sort_values(["id_rodovia", "data_inversa"])
    .reset_index(drop=True)
)

# 8. Validar a construção
duplicatas = df_base_diaria.duplicated(
    ["id_rodovia", "data_inversa"]
).sum()

print("=" * 80)
print("SEÇÃO 4.5 — CONSTRUÇÃO DA BASE DIÁRIA")
print("=" * 80)

print(f"\nPares UF × BR: {len(universo_rodovias):,}")
print(f"Dias no calendário: {len(calendario):,}")
print(f"Total de observações: {len(df_base_diaria):,}")
print(f"Duplicatas de UF × BR × Dia: {duplicatas:,}")

print("\n--- DISTRIBUIÇÃO DA VARIÁVEL-ALVO ---")
print(
    df_base_diaria["ocorreu_sinistro"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n--- PROPORÇÕES DA VARIÁVEL-ALVO ---")
print(
    (
        df_base_diaria["ocorreu_sinistro"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )
    .round(2)
    .to_string()
)

print("\n--- CONFERÊNCIA DOS SINISTROS ---")
print(
    "Sinistros registrados no universo selecionado: "
    f"{contagem_diaria.loc[contagem_diaria['id_rodovia'].isin(universo_rodovias['id_rodovia']), 'total_sinistros'].sum():,}"
)
print(
    "Sinistros contabilizados na base diária: "
    f"{df_base_diaria['total_sinistros'].sum():,}"
)

print("\n--- PRIMEIRAS 10 OBSERVAÇÕES ---")
print(df_base_diaria.head(10).to_string(index=False))


SEÇÃO 4.5 — CONSTRUÇÃO DA BASE DIÁRIA

Pares UF × BR: 230
Dias no calendário: 2,922
Total de observações: 672,060
Duplicatas de UF × BR × Dia: 0

--- DISTRIBUIÇÃO DA VARIÁVEL-ALVO ---
ocorreu_sinistro
0    443485
1    228575

--- PROPORÇÕES DA VARIÁVEL-ALVO ---
ocorreu_sinistro
0    65.99
1    34.01

--- CONFERÊNCIA DOS SINISTROS ---
Sinistros registrados no universo selecionado: 540,908
Sinistros contabilizados na base diária: 540,908

--- PRIMEIRAS 10 OBSERVAÇÕES ---
uf  br id_rodovia data_inversa  total_sinistros  ocorreu_sinistro
AC 317   AC_BR317   2018-01-01                0                 0
AC 317   AC_BR317   2018-01-02                0                 0
AC 317   AC_BR317   2018-01-03                0                 0
AC 317   AC_BR317   2018-01-04                0                 0
AC 317   AC_BR317   2018-01-05                0                 0
AC 317   AC_BR317   2018-01-06                0                 0
AC 317   AC_BR317   2018-01-07                1                 

### __4.6 - Validação temporal da base diária__

In [ ]:
# 1. Criar ano apenas na base de diagnóstico
df_diagnostico = df_base_diaria.copy()

df_diagnostico["ano"] = (
    df_diagnostico["data_inversa"].dt.year
)

# 2. Resumir os resultados por ano
resumo_anual = (
    df_diagnostico
    .groupby("ano")
    .agg(
        observacoes=("ocorreu_sinistro", "size"),
        pares=("id_rodovia", "nunique"),
        dias_positivos=("ocorreu_sinistro", "sum"),
        total_sinistros=("total_sinistros", "sum")
    )
    .reset_index()
)

resumo_anual["dias_negativos"] = (
    resumo_anual["observacoes"]
    - resumo_anual["dias_positivos"]
)

resumo_anual["taxa_positivos_pct"] = (
    100 * resumo_anual["dias_positivos"]
    / resumo_anual["observacoes"]
).round(2)

# 3. Verificar cobertura de cada par
cobertura_diaria = (
    df_diagnostico
    .groupby("id_rodovia")
    .agg(
        total_dias=("data_inversa", "nunique"),
        primeira_data=("data_inversa", "min"),
        ultima_data=("data_inversa", "max")
    )
    .reset_index()
)

pares_incompletos = cobertura_diaria.loc[
    cobertura_diaria["total_dias"] != len(calendario)
]

# 4. Validar consistência entre alvo e contagem
inconsistencias_alvo = (
    df_diagnostico["ocorreu_sinistro"]
    != (df_diagnostico["total_sinistros"] > 0).astype(int)
).sum()

# 5. Exibir resultados
print("=" * 80)
print("SEÇÃO 4.6 — VALIDAÇÃO TEMPORAL DA BASE DIÁRIA")
print("=" * 80)

print("\n--- DISTRIBUIÇÃO ANUAL ---")
print(resumo_anual.to_string(index=False))

print("\n--- COBERTURA DOS PARES UF × BR ---")
print(f"Pares analisados: {len(cobertura_diaria):,}")
print(f"Dias esperados por par: {len(calendario):,}")
print(f"Pares com calendário incompleto: {len(pares_incompletos):,}")

print("\n--- INTEGRIDADE DA VARIÁVEL-ALVO ---")
print(f"Inconsistências encontradas: {inconsistencias_alvo:,}")

print("\n--- VARIAÇÃO ANUAL DA TAXA DE POSITIVOS ---")
print(
    f"Menor taxa: {resumo_anual['taxa_positivos_pct'].min():.2f}%"
)
print(
    f"Maior taxa: {resumo_anual['taxa_positivos_pct'].max():.2f}%"
)
print(
    "Diferença: "
    f"{resumo_anual['taxa_positivos_pct'].max() - resumo_anual['taxa_positivos_pct'].min():.2f}"
    " pontos percentuais"
)

# 6. Encerrar diagnóstico sem modificar a base principal
del df_diagnostico


SEÇÃO 4.6 — VALIDAÇÃO TEMPORAL DA BASE DIÁRIA

--- DISTRIBUIÇÃO ANUAL ---
 ano  observacoes  pares  dias_positivos  total_sinistros  dias_negativos  taxa_positivos_pct
2018        83950    230           28631            69199           55319               34.10
2019        83950    230           28485            67445           55465               33.93
2020        84180    230           28000            63390           56180               33.26
2021        83950    230           27892            64294           56058               33.22
2022        83950    230           27995            64309           55955               33.35
2023        83950    230           28536            67370           55414               33.99
2024        84180    230           29784            72758           54396               35.38
2025        83950    230           29252            72143           54698               34.84

--- COBERTURA DOS PARES UF × BR ---
Pares analisados: 230
Dias esperados por pa

## __5 - Feature Engineering__

### __5.1 — Criação dos atributos temporais__

In [ ]:
# 1. Criar uma cópia da base diária
df_modelagem = df_base_diaria.copy()

# 2. Extrair atributos do calendário
datas = df_modelagem["data_inversa"]

df_modelagem["ano"] = datas.dt.year
df_modelagem["mes"] = datas.dt.month
df_modelagem["dia_semana"] = datas.dt.dayofweek
df_modelagem["dia_ano"] = datas.dt.dayofyear
df_modelagem["fim_semana"] = (
    datas.dt.dayofweek >= 5
).astype(int)
df_modelagem["trimestre"] = datas.dt.quarter

# 3. Conferir atributos criados
atributos_temporais = [
    "ano", "mes", "dia_semana",
    "dia_ano", "fim_semana", "trimestre"
]

print("=" * 80)
print("SEÇÃO 5.1 — ATRIBUTOS TEMPORAIS")
print("=" * 80)

print(f"\nObservações: {len(df_modelagem):,}")
print(f"Atributos criados: {len(atributos_temporais)}")

print("\n--- VALORES AUSENTES ---")
print(
    df_modelagem[atributos_temporais]
    .isna()
    .sum()
    .to_string()
)

print("\n--- PRIMEIRAS 10 OBSERVAÇÕES ---")
colunas_exibicao = [
    "id_rodovia", "data_inversa",
    *atributos_temporais,
    "ocorreu_sinistro"
]

print(
    df_modelagem[colunas_exibicao]
    .head(10)
    .to_string(index=False)
)

print("\n--- DIMENSÕES DA BASE ---")
print(f"Linhas: {df_modelagem.shape[0]:,}")
print(f"Colunas: {df_modelagem.shape[1]}")


SEÇÃO 5.1 — ATRIBUTOS TEMPORAIS

Observações: 672,060
Atributos criados: 6

--- VALORES AUSENTES ---
ano           0
mes           0
dia_semana    0
dia_ano       0
fim_semana    0
trimestre     0

--- PRIMEIRAS 10 OBSERVAÇÕES ---
id_rodovia data_inversa  ano  mes  dia_semana  dia_ano  fim_semana  trimestre  ocorreu_sinistro
  AC_BR317   2018-01-01 2018    1           0        1           0          1                 0
  AC_BR317   2018-01-02 2018    1           1        2           0          1                 0
  AC_BR317   2018-01-03 2018    1           2        3           0          1                 0
  AC_BR317   2018-01-04 2018    1           3        4           0          1                 0
  AC_BR317   2018-01-05 2018    1           4        5           0          1                 0
  AC_BR317   2018-01-06 2018    1           5        6           1          1                 0
  AC_BR317   2018-01-07 2018    1           6        7           1          1                 1
 

### __5.2 — Construção dos atributos históricos__

In [ ]:
# 1. Recuperar o histórico de dezembro de 2017 para inicializar as janelas móveis de 2018

historico_2017 = (
    df_uf_br.loc[
        df_uf_br["data_inversa"].between(
            "2017-12-02", "2017-12-31"
        )
        & df_uf_br["id_rodovia"].isin(
            universo_rodovias["id_rodovia"]
        )
    ]
    .groupby(["id_rodovia", "data_inversa"])
    .size()
    .reset_index(name="total_sinistros")
)

# 2. Criar calendário completo dos 30 dias anteriores
calendario_historico = pd.DataFrame({
    "data_inversa": pd.date_range(
        "2017-12-02", "2017-12-31", freq="D"
    )
})

base_historica = (
    universo_rodovias[["id_rodovia"]]
    .merge(calendario_historico, how="cross")
    .merge(
        historico_2017,
        on=["id_rodovia", "data_inversa"],
        how="left"
    )
)

base_historica["total_sinistros"] = (
    base_historica["total_sinistros"]
    .fillna(0)
    .astype(int)
)

base_historica["ocorreu_sinistro"] = (
    base_historica["total_sinistros"] > 0
).astype(int)

# 3. Combinar histórico inicial com a base de modelagem
colunas_historicas = [
    "id_rodovia", "data_inversa",
    "total_sinistros", "ocorreu_sinistro"
]

base_calculo = pd.concat(
    [
        base_historica[colunas_historicas],
        df_modelagem[colunas_historicas]
    ],
    ignore_index=True
).sort_values(
    ["id_rodovia", "data_inversa"]
).reset_index(drop=True)

# 4. Criar atributos históricos sem vazamento temporal
atributos_historicos = []

for janela in [7, 14, 30]:

    coluna_total = f"sinistros_ultimos_{janela}d"
    coluna_dias = f"dias_com_sinistro_ultimos_{janela}d"

    base_calculo[coluna_total] = (
        base_calculo
        .groupby("id_rodovia")["total_sinistros"]
        .transform(
            lambda serie: (
                serie.shift(1)
                .rolling(window=janela, min_periods=janela)
                .sum()
            )
        )
    )

    base_calculo[coluna_dias] = (
        base_calculo
        .groupby("id_rodovia")["ocorreu_sinistro"]
        .transform(
            lambda serie: (
                serie.shift(1)
                .rolling(window=janela, min_periods=janela)
                .sum()
            )
        )
    )

    atributos_historicos.extend(
        [coluna_total, coluna_dias]
    )

# 5. Incorporar atributos à base principal
novos_atributos = base_calculo.loc[
    base_calculo["data_inversa"] >= "2018-01-01",
    ["id_rodovia", "data_inversa"] + atributos_historicos
]

df_modelagem = df_modelagem.merge(
    novos_atributos,
    on=["id_rodovia", "data_inversa"],
    how="left",
    validate="one_to_one"
)

# 6. Validar resultados
print("=" * 80)
print("SEÇÃO 5.2 — ATRIBUTOS HISTÓRICOS")
print("=" * 80)

print(f"\nObservações: {len(df_modelagem):,}")
print(f"Atributos históricos criados: {len(atributos_historicos)}")

print("\n--- VALORES AUSENTES ---")
print(
    df_modelagem[atributos_historicos]
    .isna()
    .sum()
    .to_string()
)

print("\n--- ESTATÍSTICAS DOS ATRIBUTOS ---")
print(
    df_modelagem[atributos_historicos]
    .describe()
    .round(2)
    .to_string()
)

print("\n--- EXEMPLO: PRIMEIROS DIAS DE 2018 ---")
print(
    df_modelagem.loc[
        df_modelagem["id_rodovia"] == "PB_BR230",
        [
            "data_inversa",
            "total_sinistros",
            "sinistros_ultimos_7d",
            "sinistros_ultimos_30d",
            "dias_com_sinistro_ultimos_7d",
            "ocorreu_sinistro"
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\n--- DIMENSÕES DA BASE ---")
print(f"Linhas: {df_modelagem.shape[0]:,}")
print(f"Colunas: {df_modelagem.shape[1]}")


SEÇÃO 5.2 — ATRIBUTOS HISTÓRICOS

Observações: 672,060
Atributos históricos criados: 6

--- VALORES AUSENTES ---
sinistros_ultimos_7d             0
dias_com_sinistro_ultimos_7d     0
sinistros_ultimos_14d            0
dias_com_sinistro_ultimos_14d    0
sinistros_ultimos_30d            0
dias_com_sinistro_ultimos_30d    0

--- ESTATÍSTICAS DOS ATRIBUTOS ---
       sinistros_ultimos_7d  dias_com_sinistro_ultimos_7d  sinistros_ultimos_14d  dias_com_sinistro_ultimos_14d  sinistros_ultimos_30d  dias_com_sinistro_ultimos_30d
count             672060.00                     672060.00              672060.00                      672060.00              672060.00                      672060.00
mean                   5.64                          2.38                  11.28                           4.76                  24.19                          10.21
std                   10.33                          2.40                  20.37                           4.61                  43.32         

### __5.3 - Atributos de defasagem (lags)__

In [ ]:
# 1. Criar cópia da base de cálculo anterior
base_lags = base_calculo[
    ["id_rodovia", "data_inversa", "ocorreu_sinistro"]
].copy()

# 2. Criar defasagens por rodovia
atributos_lags = []

for lag in [1, 7, 14, 30]:
    coluna = f"sinistro_lag_{lag}d"

    base_lags[coluna] = (
        base_lags
        .groupby("id_rodovia")["ocorreu_sinistro"]
        .shift(lag)
    )

    atributos_lags.append(coluna)

# 3. Selecionar apenas o período de modelagem
novos_lags = base_lags.loc[
    base_lags["data_inversa"] >= "2018-01-01",
    ["id_rodovia", "data_inversa"] + atributos_lags
]

# 4. Incorporar à base de modelagem
df_modelagem = df_modelagem.merge(
    novos_lags,
    on=["id_rodovia", "data_inversa"],
    how="left",
    validate="one_to_one"
)

# 5. Validar os resultados
print("=" * 80)
print("SEÇÃO 5.3 — ATRIBUTOS DE DEFASAGEM")
print("=" * 80)

print(f"\nObservações: {len(df_modelagem):,}")
print(f"Atributos criados: {len(atributos_lags)}")

print("\n--- VALORES AUSENTES ---")
print(
    df_modelagem[atributos_lags]
    .isna()
    .sum()
    .to_string()
)

print("\n--- DISTRIBUIÇÃO DOS ATRIBUTOS ---")
print(
    df_modelagem[atributos_lags]
    .agg(["min", "max", "mean"])
    .round(3)
    .to_string()
)

print("\n--- EXEMPLO: PB_BR230 ---")
print(
    df_modelagem.loc[
        (df_modelagem["id_rodovia"] == "PB_BR230")
        & df_modelagem["data_inversa"].between(
            "2018-01-01", "2018-01-10"
        ),
        [
            "data_inversa",
            "sinistro_lag_1d",
            "sinistro_lag_7d",
            "sinistro_lag_14d",
            "sinistro_lag_30d",
            "ocorreu_sinistro"
        ]
    ].to_string(index=False)
)

print("\n--- DIMENSÕES DA BASE ---")
print(f"Linhas: {df_modelagem.shape[0]:,}")
print(f"Colunas: {df_modelagem.shape[1]}")


SEÇÃO 5.3 — ATRIBUTOS DE DEFASAGEM

Observações: 672,060
Atributos criados: 4

--- VALORES AUSENTES ---
sinistro_lag_1d     0
sinistro_lag_7d     0
sinistro_lag_14d    0
sinistro_lag_30d    0

--- DISTRIBUIÇÃO DOS ATRIBUTOS ---
      sinistro_lag_1d  sinistro_lag_7d  sinistro_lag_14d  sinistro_lag_30d
min              0.00             0.00              0.00              0.00
max              1.00             1.00              1.00              1.00
mean             0.34             0.34              0.34              0.34

--- EXEMPLO: PB_BR230 ---
data_inversa  sinistro_lag_1d  sinistro_lag_7d  sinistro_lag_14d  sinistro_lag_30d  ocorreu_sinistro
  2018-01-01              1.0              1.0               1.0               1.0                 1
  2018-01-02              1.0              1.0               1.0               1.0                 1
  2018-01-03              1.0              1.0               1.0               1.0                 0
  2018-01-04              0.0            

### __5.4 - Feriados nacionais__

In [ ]:
import holidays

# 1. Criar calendário de feriados brasileiros
anos = range(2018, 2026)

feriados_br = holidays.Brazil(
    years=anos,
    subdiv=None,
    observed=False
)

# 2. Manter apenas feriados nacionais
datas_feriados = set(feriados_br.keys())

df_modelagem["feriado_nacional"] = (
    df_modelagem["data_inversa"]
    .dt.date
    .isin(datas_feriados)
    .astype(int)
)

# 3. Validar resultados
print("=" * 80)
print("SEÇÃO 5.4 — FERIADOS NACIONAIS")
print("=" * 80)

print("\n--- DISTRIBUIÇÃO ---")
print(
    df_modelagem["feriado_nacional"]
    .value_counts()
    .to_string()
)

print("\n--- FERIADOS IDENTIFICADOS POR ANO ---")
resumo_feriados = (
    df_modelagem.loc[
        df_modelagem["feriado_nacional"] == 1,
        ["data_inversa", "ano"]
    ]
    .drop_duplicates()
    .groupby("ano")
    .size()
    .reset_index(name="total_feriados")
)

print(resumo_feriados.to_string(index=False))

print("\n--- EXEMPLOS DE FERIADOS ---")
print(
    df_modelagem.loc[
        df_modelagem["feriado_nacional"] == 1,
        ["data_inversa"]
    ]
    .drop_duplicates()
    .head(15)
    .to_string(index=False)
)

print("\n--- VALORES AUSENTES ---")
print(
    df_modelagem["feriado_nacional"]
    .isna()
    .sum()
)

print("\n--- DIMENSÕES DA BASE ---")
print(f"Linhas: {df_modelagem.shape[0]:,}")
print(f"Colunas: {df_modelagem.shape[1]}")


SEÇÃO 5.4 — FERIADOS NACIONAIS

--- DISTRIBUIÇÃO ---
feriado_nacional
0    655040
1     17020

--- FERIADOS IDENTIFICADOS POR ANO ---
 ano  total_feriados
2018               9
2019               9
2020               9
2021               9
2022               9
2023               9
2024              10
2025              10

--- EXEMPLOS DE FERIADOS ---
data_inversa
  2018-01-01
  2018-03-30
  2018-04-21
  2018-05-01
  2018-09-07
  2018-10-12
  2018-11-02
  2018-11-15
  2018-12-25
  2019-01-01
  2019-04-19
  2019-04-21
  2019-05-01
  2019-09-07
  2019-10-12

--- VALORES AUSENTES ---
0

--- DIMENSÕES DA BASE ---
Linhas: 672,060
Colunas: 23


## __6 - Preparação para a modelagem__

### __6.1 - Divisão temporal__

In [ ]:
# 1. Separar os períodos
df_treino = df_modelagem.loc[
    df_modelagem["ano"].between(2018, 2023)
].copy()

df_validacao = df_modelagem.loc[
    df_modelagem["ano"] == 2024
].copy()

df_teste = df_modelagem.loc[
    df_modelagem["ano"] == 2025
].copy()

# 2. Resumir cada conjunto
resumo_divisao = pd.DataFrame([
    {
        "conjunto": nome,
        "inicio": dados["data_inversa"].min().date(),
        "fim": dados["data_inversa"].max().date(),
        "observacoes": len(dados),
        "pares": dados["id_rodovia"].nunique(),
        "positivos": dados["ocorreu_sinistro"].sum(),
        "taxa_positivos_pct": round(
            100 * dados["ocorreu_sinistro"].mean(), 2
        )
    }
    for nome, dados in [
        ("Treino", df_treino),
        ("Validacao", df_validacao),
        ("Teste", df_teste)
    ]
])

# 3. Verificar se todos os registros foram distribuídos
total_dividido = (
    len(df_treino)
    + len(df_validacao)
    + len(df_teste)
)

print("=" * 80)
print("SEÇÃO 6.1 — DIVISÃO TEMPORAL DOS DADOS")
print("=" * 80)

print("\n--- RESUMO DOS CONJUNTOS ---")
print(resumo_divisao.to_string(index=False))

print("\n--- CONFERÊNCIA ---")
print(f"Total original: {len(df_modelagem):,}")
print(f"Total dividido: {total_dividido:,}")
print(
    "Todos os registros preservados:",
    total_dividido == len(df_modelagem)
)

print("\n--- INTERSEÇÃO ENTRE DATAS ---")
print(
    "Treino / Validação:",
    len(
        set(df_treino["data_inversa"])
        & set(df_validacao["data_inversa"])
    )
)
print(
    "Validação / Teste:",
    len(
        set(df_validacao["data_inversa"])
        & set(df_teste["data_inversa"])
    )
)


SEÇÃO 6.1 — DIVISÃO TEMPORAL DOS DADOS

--- RESUMO DOS CONJUNTOS ---
 conjunto     inicio        fim  observacoes  pares  positivos  taxa_positivos_pct
   Treino 2018-01-01 2023-12-31       503930    230     169539               33.64
Validacao 2024-01-01 2024-12-31        84180    230      29784               35.38
    Teste 2025-01-01 2025-12-31        83950    230      29252               34.84

--- CONFERÊNCIA ---
Total original: 672,060
Total dividido: 672,060
Todos os registros preservados: True

--- INTERSEÇÃO ENTRE DATAS ---
Treino / Validação: 0
Validação / Teste: 0


### __6.2 — Definição das variáveis preditoras e do alvo__

In [ ]:
# 1. Definir os grupos de atributos
atributos_categoricos = ["uf", "br"]

atributos_numericos = [
    "ano",
    "mes",
    "dia_semana",
    "dia_ano",
    "fim_semana",
    "trimestre",
    "sinistros_ultimos_7d",
    "dias_com_sinistro_ultimos_7d",
    "sinistros_ultimos_14d",
    "dias_com_sinistro_ultimos_14d",
    "sinistros_ultimos_30d",
    "dias_com_sinistro_ultimos_30d",
    "sinistro_lag_1d",
    "sinistro_lag_7d",
    "sinistro_lag_14d",
    "sinistro_lag_30d",
    "feriado_nacional"
]

atributos_modelo = (
    atributos_categoricos + atributos_numericos
)

variavel_alvo = "ocorreu_sinistro"

# 2. Criar matrizes de entrada e alvo
X_treino = df_treino[atributos_modelo].copy()
y_treino = df_treino[variavel_alvo].copy()

X_validacao = df_validacao[atributos_modelo].copy()
y_validacao = df_validacao[variavel_alvo].copy()

X_teste = df_teste[atributos_modelo].copy()
y_teste = df_teste[variavel_alvo].copy()

# 3. Converter BR para categoria
for X in [X_treino, X_validacao, X_teste]:
    X["uf"] = X["uf"].astype(str)
    X["br"] = X["br"].astype(str)

# 4. Validar os conjuntos
print("=" * 80)
print("SEÇÃO 6.2 — DEFINIÇÃO DAS VARIÁVEIS")
print("=" * 80)

print(f"\nTotal de atributos: {len(atributos_modelo)}")
print(f"Atributos categóricos: {len(atributos_categoricos)}")
print(f"Atributos numéricos: {len(atributos_numericos)}")

print("\n--- DIMENSÕES ---")
for nome, X, y in [
    ("Treino", X_treino, y_treino),
    ("Validação", X_validacao, y_validacao),
    ("Teste", X_teste, y_teste)
]:
    print(
        f"{nome}: X={X.shape}, y={y.shape}"
    )

print("\n--- VALORES AUSENTES ---")
for nome, X in [
    ("Treino", X_treino),
    ("Validação", X_validacao),
    ("Teste", X_teste)
]:
    print(
        f"{nome}: {X.isna().sum().sum():,}"
    )

print("\n--- COLUNAS PREDITORAS ---")
print(X_treino.columns.tolist())

print("\n--- DISTRIBUIÇÃO DO ALVO ---")
for nome, y in [
    ("Treino", y_treino),
    ("Validação", y_validacao),
    ("Teste", y_teste)
]:
    print(
        f"{nome}: "
        f"0={(y == 0).sum():,} | "
        f"1={(y == 1).sum():,}"
    )


SEÇÃO 6.2 — DEFINIÇÃO DAS VARIÁVEIS

Total de atributos: 19
Atributos categóricos: 2
Atributos numéricos: 17

--- DIMENSÕES ---
Treino: X=(503930, 19), y=(503930,)
Validação: X=(84180, 19), y=(84180,)
Teste: X=(83950, 19), y=(83950,)

--- VALORES AUSENTES ---
Treino: 0
Validação: 0
Teste: 0

--- COLUNAS PREDITORAS ---
['uf', 'br', 'ano', 'mes', 'dia_semana', 'dia_ano', 'fim_semana', 'trimestre', 'sinistros_ultimos_7d', 'dias_com_sinistro_ultimos_7d', 'sinistros_ultimos_14d', 'dias_com_sinistro_ultimos_14d', 'sinistros_ultimos_30d', 'dias_com_sinistro_ultimos_30d', 'sinistro_lag_1d', 'sinistro_lag_7d', 'sinistro_lag_14d', 'sinistro_lag_30d', 'feriado_nacional']

--- DISTRIBUIÇÃO DO ALVO ---
Treino: 0=334,391 | 1=169,539
Validação: 0=54,396 | 1=29,784
Teste: 0=54,698 | 1=29,252


## __7 — Desenvolvimento dos modelos__

### __7.1 — Regressão logística__

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# 1. Definir pré-processamento
preprocessador = ColumnTransformer(
    transformers=[
        (
            "numericos",
            StandardScaler(),
            atributos_numericos
        ),
        (
            "categoricos",
            OneHotEncoder(handle_unknown="ignore"),
            atributos_categoricos
        )
    ]
)

# 2. Criar pipeline
modelo_logistico = Pipeline(
    steps=[
        ("preprocessamento", preprocessador),
        (
            "classificador",
            LogisticRegression(
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

# 3. Treinar somente com 2018–2023
modelo_logistico.fit(X_treino, y_treino)

# 4. Prever probabilidades para 2024
probabilidades_validacao = (
    modelo_logistico.predict_proba(X_validacao)[:, 1]
)

# 5. Converter probabilidades em classes
limiar = 0.5

previsoes_validacao = (
    probabilidades_validacao >= limiar
).astype(int)

# 6. Calcular métricas
pr_auc = average_precision_score(
    y_validacao,
    probabilidades_validacao
)

roc_auc = roc_auc_score(
    y_validacao,
    probabilidades_validacao
)

precisao = precision_score(
    y_validacao,
    previsoes_validacao,
    zero_division=0
)

revocacao = recall_score(
    y_validacao,
    previsoes_validacao,
    zero_division=0
)

f1 = f1_score(
    y_validacao,
    previsoes_validacao,
    zero_division=0
)

matriz = confusion_matrix(
    y_validacao,
    previsoes_validacao
)

# 7. Exibir resultados
print("=" * 80)
print("SEÇÃO 7.1 — BASELINE: REGRESSÃO LOGÍSTICA")
print("=" * 80)

print("\n--- MÉTRICAS DE VALIDAÇÃO (2024) ---")
print(f"PR-AUC (Average Precision): {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Precisão: {precisao:.4f}")
print(f"Revocação: {revocacao:.4f}")
print(f"F1-score: {f1:.4f}")

print("\n--- MATRIZ DE CONFUSÃO ---")
print(matriz)

print("\n--- REFERÊNCIA: PREVALÊNCIA DA CLASSE POSITIVA ---")
print(f"Taxa de positivos: {y_validacao.mean():.4f}")

print("\n--- DISTRIBUIÇÃO DAS PROBABILIDADES ---")
print(
    pd.Series(probabilidades_validacao)
    .describe()
    .round(4)
    .to_string()
)


SEÇÃO 7.1 — BASELINE: REGRESSÃO LOGÍSTICA

--- MÉTRICAS DE VALIDAÇÃO (2024) ---
PR-AUC (Average Precision): 0.8308
ROC-AUC: 0.8835
Precisão: 0.7942
Revocação: 0.6485
F1-score: 0.7140

--- MATRIZ DE CONFUSÃO ---
[[49392  5004]
 [10470 19314]]

--- REFERÊNCIA: PREVALÊNCIA DA CLASSE POSITIVA ---
Taxa de positivos: 0.3538

--- DISTRIBUIÇÃO DAS PROBABILIDADES ---
count    84180.0000
mean         0.3515
std          0.3189
min          0.0018
25%          0.0897
50%          0.2231
75%          0.5831
max          1.0000


### __7.2 — Avaliação de diferentes limiares__

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

resultados_limiares = []

for limiar in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:

    previsoes = (
        probabilidades_validacao >= limiar
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_validacao, previsoes, labels=[0, 1]
    ).ravel()

    resultados_limiares.append({
        "limiar": limiar,
        "precisao": precision_score(
            y_validacao, previsoes, zero_division=0
        ),
        "revocacao": recall_score(
            y_validacao, previsoes, zero_division=0
        ),
        "f1_score": f1_score(
            y_validacao, previsoes, zero_division=0
        ),
        "falsos_positivos": fp,
        "falsos_negativos": fn
    })

df_limiares = pd.DataFrame(resultados_limiares)

print("=" * 80)
print("SEÇÃO 7.2 — AVALIAÇÃO DOS LIMIARES")
print("=" * 80)

print("\n--- RESULTADOS POR LIMIAR ---")
print(
    df_limiares.round(4).to_string(index=False)
)

print("\n--- MELHOR LIMIAR POR F1-SCORE ---")
melhor_limiar = df_limiares.loc[
    df_limiares["f1_score"].idxmax()
]

print(melhor_limiar.to_string())

print("\n--- MÉTRICAS INDEPENDENTES DO LIMIAR ---")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"ROC-AUC: {roc_auc:.4f}")


SEÇÃO 7.2 — AVALIAÇÃO DOS LIMIARES

--- RESULTADOS POR LIMIAR ---
 limiar  precisao  revocacao  f1_score  falsos_positivos  falsos_negativos
    0.2    0.5955     0.8859    0.7122             17926              3398
    0.3    0.6735     0.8072    0.7343             11653              5743
    0.4    0.7381     0.7233    0.7306              7644              8241
    0.5    0.7942     0.6485    0.7140              5004             10470
    0.6    0.8392     0.5765    0.6835              3289             12613
    0.7    0.8858     0.4991    0.6385              1916             14918

--- MELHOR LIMIAR POR F1-SCORE ---
limiar                  0.300000
precisao                0.673531
revocacao               0.807178
f1_score                0.734323
falsos_positivos    11653.000000
falsos_negativos     5743.000000

--- MÉTRICAS INDEPENDENTES DO LIMIAR ---
PR-AUC: 0.8308
ROC-AUC: 0.8835


### __7.3: Baseline histórico simples__

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

# 1. Probabilidade estimada pela frequência histórica
probabilidades_historico = (
    df_validacao["dias_com_sinistro_ultimos_30d"].to_numpy()
    / 30
)

# 2. Comparar métricas independentes de limiar
resultados_baselines = pd.DataFrame([
    {
        "modelo": "Histórico 30 dias",
        "pr_auc": average_precision_score(
            y_validacao, probabilidades_historico
        ),
        "roc_auc": roc_auc_score(
            y_validacao, probabilidades_historico
        )
    },
    {
        "modelo": "Regressão Logística",
        "pr_auc": average_precision_score(
            y_validacao, probabilidades_validacao
        ),
        "roc_auc": roc_auc_score(
            y_validacao, probabilidades_validacao
        )
    }
])

# 3. Comparar os dois modelos com limiar 0.30
resultados_classificacao = []

for nome, probabilidades in [
    ("Histórico 30 dias", probabilidades_historico),
    ("Regressão Logística", probabilidades_validacao)
]:
    previsoes = (probabilidades >= 0.30).astype(int)

    resultados_classificacao.append({
        "modelo": nome,
        "precisao": precision_score(
            y_validacao, previsoes, zero_division=0
        ),
        "revocacao": recall_score(
            y_validacao, previsoes, zero_division=0
        ),
        "f1_score": f1_score(
            y_validacao, previsoes, zero_division=0
        )
    })

df_comparacao_baselines = pd.DataFrame(
    resultados_classificacao
)

# 4. Exibir resultados
print("=" * 80)
print("SEÇÃO 7.3 — COMPARAÇÃO COM BASELINE HISTÓRICO")
print("=" * 80)

print("\n--- MÉTRICAS INDEPENDENTES DE LIMIAR ---")
print(
    resultados_baselines
    .round(4)
    .to_string(index=False)
)

print("\n--- CLASSIFICAÇÃO COM LIMIAR 0.30 ---")
print(
    df_comparacao_baselines
    .round(4)
    .to_string(index=False)
)

print("\n--- DIFERENÇA DE PR-AUC ---")
diferenca = (
    resultados_baselines.loc[
        resultados_baselines["modelo"] == "Regressão Logística",
        "pr_auc"
    ].iloc[0]
    -
    resultados_baselines.loc[
        resultados_baselines["modelo"] == "Histórico 30 dias",
        "pr_auc"
    ].iloc[0]
)

print(f"Regressão Logística - Histórico: {diferenca:+.4f}")


SEÇÃO 7.3 — COMPARAÇÃO COM BASELINE HISTÓRICO

--- MÉTRICAS INDEPENDENTES DE LIMIAR ---
             modelo  pr_auc  roc_auc
  Histórico 30 dias  0.8176   0.8810
Regressão Logística  0.8308   0.8835

--- CLASSIFICAÇÃO COM LIMIAR 0.30 ---
             modelo  precisao  revocacao  f1_score
  Histórico 30 dias    0.6349     0.8389    0.7228
Regressão Logística    0.6735     0.8072    0.7343

--- DIFERENÇA DE PR-AUC ---
Regressão Logística - Histórico: +0.0132


### __7.4: XGBoost__

In [ ]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

# 1. Pré-processamento - Árvores não precisam de padronização numérica.
preprocessador_xgb = ColumnTransformer(
    transformers=[
        (
            "numericos",
            "passthrough",
            atributos_numericos
        ),
        (
            "categoricos",
            OneHotEncoder(handle_unknown="ignore"),
            atributos_categoricos
        )
    ]
)

# 2. Criando modelo
modelo_xgb = Pipeline(
    steps=[
        ("preprocessamento", preprocessador_xgb),
        (
            "classificador",
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="binary:logistic",
                eval_metric="logloss",
                tree_method="hist",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

# 3. Treinar com 2018–2023
modelo_xgb.fit(X_treino, y_treino)

# 4. Prever probabilidades para 2024
probabilidades_xgb = (
    modelo_xgb.predict_proba(X_validacao)[:, 1]
)

# 5. Classificação inicial com limiar 0.30
limiar_xgb = 0.30

previsoes_xgb = (
    probabilidades_xgb >= limiar_xgb
).astype(int)

# 6. Calcular métricas
pr_auc_xgb = average_precision_score(
    y_validacao, probabilidades_xgb
)

roc_auc_xgb = roc_auc_score(
    y_validacao, probabilidades_xgb
)

precisao_xgb = precision_score(
    y_validacao, previsoes_xgb, zero_division=0
)

revocacao_xgb = recall_score(
    y_validacao, previsoes_xgb, zero_division=0
)

f1_xgb = f1_score(
    y_validacao, previsoes_xgb, zero_division=0
)

f2_xgb = fbeta_score(
    y_validacao, previsoes_xgb,
    beta=2, zero_division=0
)

matriz_xgb = confusion_matrix(
    y_validacao, previsoes_xgb
)

# 7. Exibir resultados
print("=" * 80)
print("SEÇÃO 7.4 — MODELO XGBOOST")
print("=" * 80)

print("\n--- MÉTRICAS DE VALIDAÇÃO (2024) ---")
print(f"PR-AUC: {pr_auc_xgb:.4f}")
print(f"ROC-AUC: {roc_auc_xgb:.4f}")
print(f"Precisão: {precisao_xgb:.4f}")
print(f"Revocação: {revocacao_xgb:.4f}")
print(f"F1-score: {f1_xgb:.4f}")
print(f"F2-score: {f2_xgb:.4f}")

print("\n--- MATRIZ DE CONFUSÃO ---")
print(matriz_xgb)

print("\n--- COMPARAÇÃO DE PR-AUC ---")
print("Histórico 30 dias: 0.8176")
print("Regressão Logística: 0.8308")
print(f"XGBoost: {pr_auc_xgb:.4f}")

print("\n--- DISTRIBUIÇÃO DAS PROBABILIDADES ---")
print(
    pd.Series(probabilidades_xgb)
    .describe()
    .round(4)
    .to_string()
)


SEÇÃO 7.4 — MODELO XGBOOST

--- MÉTRICAS DE VALIDAÇÃO (2024) ---
PR-AUC: 0.8330
ROC-AUC: 0.8852
Precisão: 0.6645
Revocação: 0.8153
F1-score: 0.7322
F2-score: 0.7799

--- MATRIZ DE CONFUSÃO ---
[[42134 12262]
 [ 5501 24283]]

--- COMPARAÇÃO DE PR-AUC ---
Histórico 30 dias: 0.8176
Regressão Logística: 0.8308
XGBoost: 0.8330

--- DISTRIBUIÇÃO DAS PROBABILIDADES ---
count    84180.0000
mean         0.3505
std          0.3171
min          0.0033
25%          0.0779
50%          0.2438
75%          0.5746
max          0.9988


### __7.5 — Otimização dos limiares com F2-score__

In [ ]:
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

# 1. Reunir as probabilidades dos modelos
modelos_probabilidades = {
    "Histórico 30 dias": probabilidades_historico,
    "Regressão Logística": probabilidades_validacao,
    "XGBoost": probabilidades_xgb
}

# 2. Definir os limiares avaliados
limiares = np.arange(0.05, 0.51, 0.01)

resultados = []

# 3. Avaliar cada modelo e limiar
for nome, probabilidades in modelos_probabilidades.items():

    for limiar in limiares:

        previsoes = (
            probabilidades >= limiar
        ).astype(int)

        tn, fp, fn, tp = confusion_matrix(
            y_validacao,
            previsoes,
            labels=[0, 1]
        ).ravel()

        resultados.append({
            "modelo": nome,
            "limiar": round(float(limiar), 2),
            "precisao": precision_score(
                y_validacao, previsoes, zero_division=0
            ),
            "revocacao": recall_score(
                y_validacao, previsoes, zero_division=0
            ),
            "f1_score": f1_score(
                y_validacao, previsoes, zero_division=0
            ),
            "f2_score": fbeta_score(
                y_validacao,
                previsoes,
                beta=2,
                zero_division=0
            ),
            "falsos_positivos": fp,
            "falsos_negativos": fn
        })

df_resultados_limiares = pd.DataFrame(resultados)

# 4. Selecionar o melhor limiar de cada modelo
indices_melhores = (
    df_resultados_limiares
    .groupby("modelo")["f2_score"]
    .idxmax()
)

df_melhores_limiares = (
    df_resultados_limiares
    .loc[indices_melhores]
    .sort_values("f2_score", ascending=False)
    .reset_index(drop=True)
)

# 5. Exibir resultados
print("=" * 80)
print("SEÇÃO 7.5 — OTIMIZAÇÃO DOS LIMIARES POR F2-SCORE")
print("=" * 80)

print("\n--- MELHOR LIMIAR POR MODELO ---")
print(
    df_melhores_limiares
    .round(4)
    .to_string(index=False)
)

print("\n--- MELHOR MODELO PELO F2-SCORE ---")
melhor_modelo = df_melhores_limiares.iloc[0]

print(f"Modelo: {melhor_modelo['modelo']}")
print(f"Limiar: {melhor_modelo['limiar']:.2f}")
print(f"F2-score: {melhor_modelo['f2_score']:.4f}")
print(f"Precisão: {melhor_modelo['precisao']:.4f}")
print(f"Revocação: {melhor_modelo['revocacao']:.4f}")
print(
    "Falsos negativos:",
    int(melhor_modelo["falsos_negativos"])
)


SEÇÃO 7.5 — OTIMIZAÇÃO DOS LIMIARES POR F2-SCORE

--- MELHOR LIMIAR POR MODELO ---
             modelo  limiar  precisao  revocacao  f1_score  f2_score  falsos_positivos  falsos_negativos
            XGBoost    0.17    0.5505     0.9233    0.6898    0.8132             22453              2284
Regressão Logística    0.15    0.5465     0.9253    0.6872    0.8126             22865              2226
  Histórico 30 dias    0.14    0.5403     0.9279    0.6830    0.8115             23512              2146

--- MELHOR MODELO PELO F2-SCORE ---
Modelo: XGBoost
Limiar: 0.17
F2-score: 0.8132
Precisão: 0.5505
Revocação: 0.9233
Falsos negativos: 2284


### __7.6 - Importância dos atributos do XGBoost__

In [ ]:
# 1. Recuperar o pré-processador e o classificador treinados
preprocessador_treinado = modelo_xgb.named_steps[
    "preprocessamento"
]

classificador_treinado = modelo_xgb.named_steps[
    "classificador"
]

# 2. Recuperar os nomes dos atributos transformados
nomes_atributos = (
    preprocessador_treinado.get_feature_names_out()
)

# 3. Recuperar a importância dos atributos
importancias = classificador_treinado.feature_importances_

# 4. Construir tabela de importância
df_importancias = pd.DataFrame({
    "atributo": nomes_atributos,
    "importancia": importancias
})

# 5. Remover prefixos gerados pelo ColumnTransformer
df_importancias["atributo"] = (
    df_importancias["atributo"]
    .str.replace("numericos__", "", regex=False)
    .str.replace("categoricos__", "", regex=False)
)

# 6. Ordenar por importância
df_importancias = (
    df_importancias
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)

# 7. Exibir resultados
print("=" * 80)
print("SEÇÃO 7.6 — IMPORTÂNCIA DOS ATRIBUTOS")
print("=" * 80)

print("\n--- TOP 20 ATRIBUTOS MAIS IMPORTANTES ---")
print(
    df_importancias.head(20)
    .round(4)
    .to_string(index=False)
)

print("\n--- TOTAL DE ATRIBUTOS APÓS CODIFICAÇÃO ---")
print(len(df_importancias))

print("\n--- SOMA DAS IMPORTÂNCIAS ---")
print(round(df_importancias["importancia"].sum(), 4))


SEÇÃO 7.6 — IMPORTÂNCIA DOS ATRIBUTOS

--- TOP 20 ATRIBUTOS MAIS IMPORTANTES ---
                     atributo  importancia
        sinistros_ultimos_30d       0.3794
        sinistros_ultimos_14d       0.1096
dias_com_sinistro_ultimos_30d       0.0836
                       br_428       0.0078
                       br_267       0.0074
                        br_60       0.0070
                       br_156       0.0070
                   fim_semana       0.0068
                        uf_MG       0.0066
                   dia_semana       0.0063
                        uf_SE       0.0063
         sinistros_ultimos_7d       0.0061
                       br_471       0.0061
                       br_415       0.0061
                        br_10       0.0059
                        br_70       0.0058
                       br_251       0.0055
                       br_104       0.0052
                       br_401       0.0049
                       br_356       0.0049

--- TOTAL DE AT

## __Seção 8 - Avaliação final dos modelos__

### __8.1 - Avaliação no conjunto de teste__

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score,
    confusion_matrix
)

# 1. Probabilidades do baseline histórico
prob_historico_teste = (
    df_teste["dias_com_sinistro_ultimos_30d"].to_numpy()
    / 30
)

# 2. Probabilidades da regressão logística
prob_logistica_teste = (
    modelo_logistico.predict_proba(X_teste)[:, 1]
)

# 3. Probabilidades do XGBoost
prob_xgb_teste = (
    modelo_xgb.predict_proba(X_teste)[:, 1]
)

# 4. Modelos e limiares selecionados em 2024
modelos_teste = {
    "Histórico 30 dias": (prob_historico_teste, 0.14),
    "Regressão Logística": (prob_logistica_teste, 0.15),
    "XGBoost": (prob_xgb_teste, 0.17)
}

resultados_teste = []

# 5. Avaliar os modelos
for nome, (probabilidades, limiar) in modelos_teste.items():

    previsoes = (
        probabilidades >= limiar
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_teste, previsoes, labels=[0, 1]
    ).ravel()

    resultados_teste.append({
        "modelo": nome,
        "limiar": limiar,
        "pr_auc": average_precision_score(
            y_teste, probabilidades
        ),
        "roc_auc": roc_auc_score(
            y_teste, probabilidades
        ),
        "precisao": precision_score(
            y_teste, previsoes, zero_division=0
        ),
        "revocacao": recall_score(
            y_teste, previsoes, zero_division=0
        ),
        "f1_score": f1_score(
            y_teste, previsoes, zero_division=0
        ),
        "f2_score": fbeta_score(
            y_teste, previsoes,
            beta=2, zero_division=0
        ),
        "verdadeiros_positivos": tp,
        "falsos_positivos": fp,
        "falsos_negativos": fn,
        "verdadeiros_negativos": tn
    })

df_resultados_teste = pd.DataFrame(resultados_teste)

# 6. Exibir resultados
print("=" * 80)
print("SEÇÃO 8.1 — AVALIAÇÃO FINAL DOS MODELOS (2025)")
print("=" * 80)

print("\n--- MÉTRICAS DE TESTE ---")
print(
    df_resultados_teste[
        [
            "modelo",
            "limiar",
            "pr_auc",
            "roc_auc",
            "precisao",
            "revocacao",
            "f1_score",
            "f2_score"
        ]
    ].round(4).to_string(index=False)
)

print("\n--- MATRIZES DE CONFUSÃO (RESUMO) ---")
print(
    df_resultados_teste[
        [
            "modelo",
            "verdadeiros_positivos",
            "falsos_positivos",
            "falsos_negativos",
            "verdadeiros_negativos"
        ]
    ].to_string(index=False)
)

print("\n--- MELHOR MODELO POR PR-AUC ---")
melhor_pr_auc = df_resultados_teste.loc[
    df_resultados_teste["pr_auc"].idxmax()
]
print(
    f"{melhor_pr_auc['modelo']}: "
    f"{melhor_pr_auc['pr_auc']:.4f}"
)

print("\n--- MELHOR MODELO POR F2-SCORE ---")
melhor_f2 = df_resultados_teste.loc[
    df_resultados_teste["f2_score"].idxmax()
]
print(
    f"{melhor_f2['modelo']}: "
    f"{melhor_f2['f2_score']:.4f}"
)


SEÇÃO 8.1 — AVALIAÇÃO FINAL DOS MODELOS (2025)

--- MÉTRICAS DE TESTE ---
             modelo  limiar  pr_auc  roc_auc  precisao  revocacao  f1_score  f2_score
  Histórico 30 dias    0.14  0.8121   0.8789    0.5262     0.9261    0.6711    0.8039
Regressão Logística    0.15  0.8261   0.8812    0.5359     0.9260    0.6789    0.8083
            XGBoost    0.17  0.8285   0.8833    0.5397     0.9225    0.6810    0.8079

--- MATRIZES DE CONFUSÃO (RESUMO) ---
             modelo  verdadeiros_positivos  falsos_positivos  falsos_negativos  verdadeiros_negativos
  Histórico 30 dias                  27091             24391              2161                  30307
Regressão Logística                  27087             23461              2165                  31237
            XGBoost                  26985             23011              2267                  31687

--- MELHOR MODELO POR PR-AUC ---
XGBoost: 0.8285

--- MELHOR MODELO POR F2-SCORE ---
Regressão Logística: 0.8083


### __8.2 - Consolidação dos resultados__

In [ ]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    fbeta_score
)

# 1. Modelos, probabilidades e limiares
modelos_comparacao = {
    "Histórico 30 dias": {
        "validacao": probabilidades_historico,
        "teste": prob_historico_teste,
        "limiar": 0.14
    },
    "Regressão Logística": {
        "validacao": probabilidades_validacao,
        "teste": prob_logistica_teste,
        "limiar": 0.15
    },
    "XGBoost": {
        "validacao": probabilidades_xgb,
        "teste": prob_xgb_teste,
        "limiar": 0.17
    }
}

resultados_consolidados = []

# 2. Calcular métricas para cada período
for nome, dados in modelos_comparacao.items():

    for periodo, y_real in [
        ("Validação 2024", y_validacao),
        ("Teste 2025", y_teste)
    ]:

        chave = (
            "validacao" if periodo == "Validação 2024"
            else "teste"
        )

        probabilidades = dados[chave]
        limiar = dados["limiar"]

        previsoes = (
            probabilidades >= limiar
        ).astype(int)

        resultados_consolidados.append({
            "modelo": nome,
            "periodo": periodo,
            "limiar": limiar,
            "pr_auc": average_precision_score(
                y_real, probabilidades
            ),
            "roc_auc": roc_auc_score(
                y_real, probabilidades
            ),
            "precisao": precision_score(
                y_real, previsoes, zero_division=0
            ),
            "revocacao": recall_score(
                y_real, previsoes, zero_division=0
            ),
            "f1_score": f1_score(
                y_real, previsoes, zero_division=0
            ),
            "f2_score": fbeta_score(
                y_real, previsoes,
                beta=2, zero_division=0
            )
        })

df_resultados_consolidados = pd.DataFrame(
    resultados_consolidados
)

# 3. Exibir tabela
print("=" * 80)
print("SEÇÃO 8.2 — RESULTADOS CONSOLIDADOS")
print("=" * 80)

print(
    df_resultados_consolidados
    .round(4)
    .to_string(index=False)
)

print("\n--- DIFERENÇA DE PR-AUC ENTRE TESTE E VALIDAÇÃO ---")

tabela_pr_auc = df_resultados_consolidados.pivot(
    index="modelo",
    columns="periodo",
    values="pr_auc"
)

tabela_pr_auc["diferenca"] = (
    tabela_pr_auc["Teste 2025"]
    - tabela_pr_auc["Validação 2024"]
)

print(
    tabela_pr_auc
    .round(4)
    .to_string()
)


SEÇÃO 8.2 — RESULTADOS CONSOLIDADOS
             modelo        periodo  limiar  pr_auc  roc_auc  precisao  revocacao  f1_score  f2_score
  Histórico 30 dias Validação 2024    0.14  0.8176   0.8810    0.5403     0.9279    0.6830    0.8115
  Histórico 30 dias     Teste 2025    0.14  0.8121   0.8789    0.5262     0.9261    0.6711    0.8039
Regressão Logística Validação 2024    0.15  0.8308   0.8835    0.5465     0.9253    0.6872    0.8126
Regressão Logística     Teste 2025    0.15  0.8261   0.8812    0.5359     0.9260    0.6789    0.8083
            XGBoost Validação 2024    0.17  0.8330   0.8852    0.5505     0.9233    0.6898    0.8132
            XGBoost     Teste 2025    0.17  0.8285   0.8833    0.5397     0.9225    0.6810    0.8079

--- DIFERENÇA DE PR-AUC ENTRE TESTE E VALIDAÇÃO ---
periodo              Teste 2025  Validação 2024  diferenca
modelo                                                    
Histórico 30 dias        0.8121          0.8176    -0.0055
Regressão Logística      0.

### __8.3 - Análise do desempenho por rodovia__

In [ ]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score
)

# 1. Construir base de avaliação por rodovia
df_avaliacao_rodovias = df_teste[
    ["id_rodovia", "uf", "br", "data_inversa", "ocorreu_sinistro"]
].copy()

df_avaliacao_rodovias["probabilidade"] = prob_xgb_teste

df_avaliacao_rodovias["previsao"] = (
    df_avaliacao_rodovias["probabilidade"] >= 0.17
).astype(int)

# 2. Calcular métricas por rodovia
resultados_rodovias = []

for id_rodovia, grupo in df_avaliacao_rodovias.groupby(
    "id_rodovia"
):
    y_real = grupo["ocorreu_sinistro"]
    y_previsto = grupo["previsao"]

    resultados_rodovias.append({
        "id_rodovia": id_rodovia,
        "dias_avaliados": len(grupo),
        "dias_com_sinistro": int(y_real.sum()),
        "taxa_ocorrencia": y_real.mean(),
        "precisao": precision_score(
            y_real, y_previsto, zero_division=0
        ),
        "revocacao": recall_score(
            y_real, y_previsto, zero_division=0
        ),
        "f2_score": fbeta_score(
            y_real, y_previsto,
            beta=2, zero_division=0
        ),
        "alertas_emitidos": int(y_previsto.sum())
    })

df_metricas_rodovias = pd.DataFrame(
    resultados_rodovias
)

# 3. Resumo geral
print("=" * 80)
print("SEÇÃO 8.3 — DESEMPENHO DO XGBOOST POR RODOVIA")
print("=" * 80)

print("\n--- RESUMO ---")
print(f"Rodovias avaliadas: {len(df_metricas_rodovias)}")

print("\n--- DISTRIBUIÇÃO DAS MÉTRICAS ---")
print(
    df_metricas_rodovias[
        [
            "taxa_ocorrencia",
            "precisao",
            "revocacao",
            "f2_score"
        ]
    ].describe().round(4).to_string()
)

# 4. Rodovias com maior número de dias positivos
print("\n--- TOP 10: MAIS DIAS COM SINISTROS ---")
print(
    df_metricas_rodovias
    .sort_values("dias_com_sinistro", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

# 5. Rodovias com menor número de dias positivos
print("\n--- TOP 10: MENOS DIAS COM SINISTROS ---")
print(
    df_metricas_rodovias
    .sort_values("dias_com_sinistro", ascending=True)
    .head(10)
    .round(4)
    .to_string(index=False)
)


SEÇÃO 8.3 — DESEMPENHO DO XGBOOST POR RODOVIA

--- RESUMO ---
Rodovias avaliadas: 230

--- DISTRIBUIÇÃO DAS MÉTRICAS ---
       taxa_ocorrencia  precisao  revocacao  f2_score
count         230.0000  230.0000   230.0000  230.0000
mean            0.3484    0.3466     0.5999    0.5006
std             0.3202    0.3235     0.4391    0.3833
min             0.0000    0.0000     0.0000    0.0000
25%             0.0822    0.0721     0.0520    0.0539
50%             0.2630    0.2599     0.9077    0.6109
75%             0.5767    0.5767     1.0000    0.8720
max             1.0000    1.0000     1.0000    1.0000

--- TOP 10: MAIS DIAS COM SINISTROS ---
id_rodovia  dias_avaliados  dias_com_sinistro  taxa_ocorrencia  precisao  revocacao  f2_score  alertas_emitidos
  MG_BR381             365                365           1.0000    1.0000        1.0    1.0000               365
  SC_BR101             365                365           1.0000    1.0000        1.0    1.0000               365
  SP_BR116      

### __8.4 - Desempenho por faixa de frequência__

In [ ]:
import numpy as np
import pandas as pd

# 1. Classificar rodovias pela frequência observada em 2025
df_faixas = df_metricas_rodovias.copy()

df_faixas["faixa_frequencia"] = pd.cut(
    df_faixas["taxa_ocorrencia"],
    bins=[-0.001, 0.10, 0.30, 0.60, 1.00],
    labels=[
        "Muito baixa (0-10%)",
        "Baixa (10-30%)",
        "Moderada (30-60%)",
        "Alta (60-100%)"
    ]
)

# 2. Resumo por faixa
resumo_faixas = (
    df_faixas
    .groupby("faixa_frequencia", observed=True)
    .agg(
        quantidade_rodovias=("id_rodovia", "count"),
        dias_positivos=("dias_com_sinistro", "sum"),
        media_precisao=("precisao", "mean"),
        media_revocacao=("revocacao", "mean"),
        media_f2=("f2_score", "mean")
    )
    .reset_index()
)

# 3. Recalcular a revocação apenas onde ela é definida
df_faixas["revocacao_valida"] = np.where(
    df_faixas["dias_com_sinistro"] > 0,
    df_faixas["revocacao"],
    np.nan
)

revocacao_corrigida = (
    df_faixas
    .groupby("faixa_frequencia", observed=True)[
        "revocacao_valida"
    ]
    .mean()
)

resumo_faixas["media_revocacao"] = (
    resumo_faixas["faixa_frequencia"]
    .map(revocacao_corrigida)
    .astype(float)
)

# 4. Exibir resultados
print("=" * 80)
print("SEÇÃO 8.4 — DESEMPENHO POR FAIXA DE FREQUÊNCIA")
print("=" * 80)

print("\n--- RESUMO POR FAIXA ---")
print(
    resumo_faixas
    .round(4)
    .to_string(index=False)
)

print("\n--- RODOVIAS SEM DIAS POSITIVOS EM 2025 ---")
print(
    (df_faixas["dias_com_sinistro"] == 0).sum()
)

print("\n--- REVOCACAO MEDIA CORRIGIDA ---")
print(
    df_faixas["revocacao_valida"].mean().round(4)
)

print("\n--- DISTRIBUICAO DE RODOVIAS POR FAIXA ---")
print(
    df_faixas["faixa_frequencia"]
    .value_counts()
    .sort_index()
    .to_string()
)


SEÇÃO 8.4 — DESEMPENHO POR FAIXA DE FREQUÊNCIA

--- RESUMO POR FAIXA ---
   faixa_frequencia  quantidade_rodovias  dias_positivos  media_precisao  media_revocacao  media_f2
Muito baixa (0-10%)                   69             958          0.0288           0.0314    0.0253
     Baixa (10-30%)                   62            4460          0.2003           0.6054    0.4244
  Moderada (30-60%)                   45            7122          0.4338           0.9931    0.7821
     Alta (60-100%)                   54           16712          0.8479           1.0000    0.9611

--- RODOVIAS SEM DIAS POSITIVOS EM 2025 ---
13

--- REVOCACAO MEDIA CORRIGIDA ---
0.6359

--- DISTRIBUICAO DE RODOVIAS POR FAIXA ---
faixa_frequencia
Muito baixa (0-10%)    69
Baixa (10-30%)         62
Moderada (30-60%)      45
Alta (60-100%)         54


### __8.5 — Revocação agregada por faixa__

In [ ]:
# 1. Associar cada rodovia à sua faixa de frequência
mapa_faixas = df_faixas[
    ["id_rodovia", "faixa_frequencia"]
].copy()

df_avaliacao_faixas = df_avaliacao_rodovias.merge(
    mapa_faixas,
    on="id_rodovia",
    how="left",
    validate="many_to_one"
)

# 2. Identificar acertos e erros
df_avaliacao_faixas["vp"] = (
    (df_avaliacao_faixas["ocorreu_sinistro"] == 1)
    & (df_avaliacao_faixas["previsao"] == 1)
).astype(int)

df_avaliacao_faixas["fn"] = (
    (df_avaliacao_faixas["ocorreu_sinistro"] == 1)
    & (df_avaliacao_faixas["previsao"] == 0)
).astype(int)

df_avaliacao_faixas["fp"] = (
    (df_avaliacao_faixas["ocorreu_sinistro"] == 0)
    & (df_avaliacao_faixas["previsao"] == 1)
).astype(int)

# 3. Agregar resultados por faixa
df_resultados_faixas = (
    df_avaliacao_faixas
    .groupby("faixa_frequencia", observed=True)
    .agg(
        dias_positivos=("ocorreu_sinistro", "sum"),
        verdadeiros_positivos=("vp", "sum"),
        falsos_negativos=("fn", "sum"),
        falsos_positivos=("fp", "sum")
    )
    .reset_index()
)

# 4. Calcular revocacao agregada
df_resultados_faixas["revocacao_agregada"] = (
    df_resultados_faixas["verdadeiros_positivos"]
    / df_resultados_faixas["dias_positivos"]
)

# 5. Validar consistência com os resultados gerais
print("=" * 80)
print("SEÇÃO 8.5 — REVOCACAO AGREGADA POR FAIXA")
print("=" * 80)

print("\n--- RESULTADOS POR FAIXA ---")
print(
    df_resultados_faixas
    .round(4)
    .to_string(index=False)
)

print("\n--- VALIDACAO DOS TOTAIS ---")
print(
    "Total de dias positivos:",
    df_resultados_faixas["dias_positivos"].sum()
)
print(
    "Total de verdadeiros positivos:",
    df_resultados_faixas["verdadeiros_positivos"].sum()
)
print(
    "Total de falsos negativos:",
    df_resultados_faixas["falsos_negativos"].sum()
)
print(
    "Total de falsos positivos:",
    df_resultados_faixas["falsos_positivos"].sum()
)


SEÇÃO 8.5 — REVOCACAO AGREGADA POR FAIXA

--- RESULTADOS POR FAIXA ---
   faixa_frequencia  dias_positivos  verdadeiros_positivos  falsos_negativos  falsos_positivos  revocacao_agregada
Muito baixa (0-10%)             958                     54               904               439              0.0564
     Baixa (10-30%)            4460                   3134              1326             10356              0.7027
  Moderada (30-60%)            7122                   7085                37              9218              0.9948
     Alta (60-100%)           16712                  16712                 0              2998              1.0000

--- VALIDACAO DOS TOTAIS ---
Total de dias positivos: 29252
Total de verdadeiros positivos: 26985
Total de falsos negativos: 2267
Total de falsos positivos: 23011


## __9 - Experimento complementar__

### __9.1 - Frequência Histórica das Rodovias__

In [ ]:

import pandas as pd

# 1. Calcular frequência histórica usando apenas o treinamento
df_frequencia_historica = (
    df_treino
    .groupby("id_rodovia")
    .agg(
        dias_historicos=("ocorreu_sinistro", "count"),
        dias_com_sinistro=("ocorreu_sinistro", "sum"),
        frequencia_historica=("ocorreu_sinistro", "mean")
    )
    .reset_index()
)

# 2. Classificar rodovias em faixas
df_frequencia_historica["faixa_historica"] = pd.cut(
    df_frequencia_historica["frequencia_historica"],
    bins=[-0.001, 0.10, 0.30, 0.60, 1.00],
    labels=[
        "Muito baixa (0-10%)",
        "Baixa (10-30%)",
        "Moderada (30-60%)",
        "Alta (60-100%)"
    ]
)

# 3. Resumo das frequências
print("=" * 80)
print("SEÇÃO 9.1 — FREQUÊNCIA HISTÓRICA DAS RODOVIAS")
print("=" * 80)

print("\n--- DISTRIBUIÇÃO DA FREQUÊNCIA HISTÓRICA ---")
print(
    df_frequencia_historica[
        "frequencia_historica"
    ].describe().round(4).to_string()
)

print("\n--- QUANTIDADE DE RODOVIAS POR FAIXA ---")
print(
    df_frequencia_historica[
        "faixa_historica"
    ].value_counts().sort_index().to_string()
)

print("\n--- EXEMPLOS DE RODOVIAS COM MENOR FREQUÊNCIA ---")
print(
    df_frequencia_historica
    .sort_values("frequencia_historica")
    .head(10)
    .round(4)
    .to_string(index=False)
)

print("\n--- EXEMPLOS DE RODOVIAS COM MAIOR FREQUÊNCIA ---")
print(
    df_frequencia_historica
    .sort_values("frequencia_historica", ascending=False)
    .head(10)
    .round(4)
    .to_string(index=False)
)

print("\n--- VALIDAÇÃO ---")
print(
    "Total de rodovias:",
    len(df_frequencia_historica)
)
print(
    "Rodovias sem classificação:",
    df_frequencia_historica[
        "faixa_historica"
    ].isna().sum()
)


SEÇÃO 9.1 — FREQUÊNCIA HISTÓRICA DAS RODOVIAS

--- DISTRIBUIÇÃO DA FREQUÊNCIA HISTÓRICA ---
count    230.0000
mean       0.3364
std        0.3125
min        0.0000
25%        0.0686
50%        0.2389
75%        0.5480
max        0.9995

--- QUANTIDADE DE RODOVIAS POR FAIXA ---
faixa_historica
Muito baixa (0-10%)    69
Baixa (10-30%)         65
Moderada (30-60%)      46
Alta (60-100%)         50

--- EXEMPLOS DE RODOVIAS COM MENOR FREQUÊNCIA ---
id_rodovia  dias_historicos  dias_com_sinistro  frequencia_historica     faixa_historica
  RS_BR473             2191                  0                0.0000 Muito baixa (0-10%)
  PI_BR404             2191                  0                0.0000 Muito baixa (0-10%)
  BA_BR122             2191                  1                0.0005 Muito baixa (0-10%)
  BA_BR423             2191                  1                0.0005 Muito baixa (0-10%)
  PE_BR235             2191                  1                0.0005 Muito baixa (0-10%)
  RS_BR377       

### __9.2 - Diagnóstico por faixa histórica na validação__

In [ ]:
import numpy as np
import pandas as pd

# 1. Preparar base de validação
df_validacao_faixas = df_validacao[
    ["id_rodovia", "data_inversa", "ocorreu_sinistro"]
].copy()

# 2. Adicionar probabilidades e previsões originais
df_validacao_faixas["probabilidade"] = probabilidades_xgb

df_validacao_faixas["previsao"] = (
    df_validacao_faixas["probabilidade"] >= 0.17
).astype(int)

# 3. Associar a faixa histórica de cada rodovia
df_validacao_faixas = df_validacao_faixas.merge(
    df_frequencia_historica[
        ["id_rodovia", "frequencia_historica", "faixa_historica"]
    ],
    on="id_rodovia",
    how="left",
    validate="many_to_one"
)

# 4. Identificar os resultados das previsões
df_validacao_faixas["vp"] = (
    (df_validacao_faixas["ocorreu_sinistro"] == 1)
    & (df_validacao_faixas["previsao"] == 1)
).astype(int)

df_validacao_faixas["fp"] = (
    (df_validacao_faixas["ocorreu_sinistro"] == 0)
    & (df_validacao_faixas["previsao"] == 1)
).astype(int)

df_validacao_faixas["fn"] = (
    (df_validacao_faixas["ocorreu_sinistro"] == 1)
    & (df_validacao_faixas["previsao"] == 0)
).astype(int)

# 5. Agregar resultados por faixa
df_diagnostico_faixas = (
    df_validacao_faixas
    .groupby("faixa_historica", observed=True)
    .agg(
        dias_positivos=("ocorreu_sinistro", "sum"),
        verdadeiros_positivos=("vp", "sum"),
        falsos_positivos=("fp", "sum"),
        falsos_negativos=("fn", "sum")
    )
    .reset_index()
)

# 6. Calcular precisão e revocação agregadas
df_diagnostico_faixas["precisao"] = (
    df_diagnostico_faixas["verdadeiros_positivos"]
    / (
        df_diagnostico_faixas["verdadeiros_positivos"]
        + df_diagnostico_faixas["falsos_positivos"]
    ).replace(0, np.nan)
)

df_diagnostico_faixas["revocacao"] = (
    df_diagnostico_faixas["verdadeiros_positivos"]
    / df_diagnostico_faixas["dias_positivos"].replace(0, np.nan)
)

# 7. Exibir resultados
print("=" * 80)
print("SEÇÃO 9.2 — DIAGNÓSTICO POR FAIXA HISTÓRICA (2024)")
print("=" * 80)

print("\n--- RESULTADOS COM LIMIAR GLOBAL DE 0.17 ---")
print(
    df_diagnostico_faixas
    .round(4)
    .to_string(index=False)
)

print("\n--- VALIDAÇÃO DOS TOTAIS ---")
print("Observações:", len(df_validacao_faixas))
print(
    "Rodovias sem faixa histórica:",
    df_validacao_faixas["faixa_historica"].isna().sum()
)
print(
    "Total de verdadeiros positivos:",
    df_diagnostico_faixas["verdadeiros_positivos"].sum()
)
print(
    "Total de falsos positivos:",
    df_diagnostico_faixas["falsos_positivos"].sum()
)
print(
    "Total de falsos negativos:",
    df_diagnostico_faixas["falsos_negativos"].sum()
)


SEÇÃO 9.2 — DIAGNÓSTICO POR FAIXA HISTÓRICA (2024)

--- RESULTADOS COM LIMIAR GLOBAL DE 0.17 ---
    faixa_historica  dias_positivos  verdadeiros_positivos  falsos_positivos  falsos_negativos  precisao  revocacao
Muito baixa (0-10%)            1067                     86               580               981    0.1291     0.0806
     Baixa (10-30%)            4959                   3691             10653              1268    0.2573     0.7443
  Moderada (30-60%)            7825                   7790              8853                35    0.4681     0.9955
     Alta (60-100%)           15933                  15933              2367                 0    0.8707     1.0000

--- VALIDAÇÃO DOS TOTAIS ---
Observações: 84180
Rodovias sem faixa histórica: 0
Total de verdadeiros positivos: 27500
Total de falsos positivos: 22453
Total de falsos negativos: 2284


### __9.3 - Experimento com limiares adaptativos__

In [ ]:
import itertools
import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score
)

# 1. Limiares candidatos para as faixas de menor frequência
limiares_candidatos = [0.01, 0.03, 0.05, 0.10, 0.15]

resultados_adaptativos = []

# 2. Avaliar combinações de limiares
for limiar_muito_baixa, limiar_baixa in itertools.product(
    limiares_candidatos,
    repeat=2
):

    # Limiares por faixa histórica
    mapa_limiares = {
        "Muito baixa (0-10%)": limiar_muito_baixa,
        "Baixa (10-30%)": limiar_baixa,
        "Moderada (30-60%)": 0.17,
        "Alta (60-100%)": 0.17
    }

    # Atribuir limiar para cada observação
    limiares_por_linha = (
        df_validacao_faixas["faixa_historica"]
        .astype(str)
        .map(mapa_limiares)
        .to_numpy()
    )

    # Gerar previsões
    previsoes = (
        df_validacao_faixas["probabilidade"].to_numpy()
        >= limiares_por_linha
    ).astype(int)

    y_real = df_validacao_faixas[
        "ocorreu_sinistro"
    ].to_numpy()

    # Identificar rodovias de frequência até 30%
    mascara_baixa_frequencia = (
        df_validacao_faixas["frequencia_historica"]
        .to_numpy() <= 0.30
    )

    y_baixa = y_real[mascara_baixa_frequencia]
    pred_baixa = previsoes[mascara_baixa_frequencia]

    # Calcular erros no grupo de baixa frequência
    fn_baixa = int(
        ((y_baixa == 1) & (pred_baixa == 0)).sum()
    )

    fp_baixa = int(
        ((y_baixa == 0) & (pred_baixa == 1)).sum()
    )

    # Armazenar métricas
    resultados_adaptativos.append({
        "limiar_muito_baixa": limiar_muito_baixa,
        "limiar_baixa": limiar_baixa,
        "precisao_global": precision_score(
            y_real, previsoes, zero_division=0
        ),
        "revocacao_global": recall_score(
            y_real, previsoes, zero_division=0
        ),
        "f2_global": fbeta_score(
            y_real, previsoes,
            beta=2, zero_division=0
        ),
        "revocacao_baixa": recall_score(
            y_baixa, pred_baixa, zero_division=0
        ),
        "fn_baixa": fn_baixa,
        "fp_baixa": fp_baixa
    })

df_resultados_adaptativos = pd.DataFrame(
    resultados_adaptativos
)

# 3. Ordenar pelo F2-score global
df_resultados_adaptativos = (
    df_resultados_adaptativos
    .sort_values("f2_global", ascending=False)
    .reset_index(drop=True)
)

# 4. Exibir resultados
print("=" * 80)
print("SEÇÃO 9.3 — EXPERIMENTO COM LIMIARES ADAPTATIVOS")
print("=" * 80)

print("\n--- TOP 10 COMBINAÇÕES POR F2-SCORE GLOBAL ---")
print(
    df_resultados_adaptativos
    .head(10)
    .round(4)
    .to_string(index=False)
)

print("\n--- MELHOR COMBINAÇÃO ---")
print(
    df_resultados_adaptativos
    .iloc[0]
    .round(4)
    .to_string()
)

print("\n--- REFERÊNCIA: XGBOOST COM LIMIAR GLOBAL 0.17 ---")
print("Precisão global: 0.5505")
print("Revocação global: 0.9233")
print("F2-score global: 0.8132")
print("Falsos negativos nas faixas até 30%: 2249")
print("Falsos positivos nas faixas até 30%: 11233")


SEÇÃO 9.3 — EXPERIMENTO COM LIMIARES ADAPTATIVOS

--- TOP 10 COMBINAÇÕES POR F2-SCORE GLOBAL ---
 limiar_muito_baixa  limiar_baixa  precisao_global  revocacao_global  f2_global  revocacao_baixa  fn_baixa  fp_baixa
               0.15          0.15           0.5345            0.9341     0.8126           0.6801      1928     13011
               0.15          0.10           0.5020            0.9567     0.8100           0.7916      1256     17043
               0.15          0.05           0.4864            0.9663     0.8070           0.8392       969     19175
               0.10          0.15           0.5125            0.9420     0.8067           0.7191      1693     15471
               0.15          0.03           0.4850            0.9669     0.8066           0.8423       950     19359
               0.15          0.01           0.4817            0.9673     0.8050           0.8442       939     19781
               0.10          0.10           0.4832            0.9645     0.8043     

### __9.4 - Comparação detalhada por faixa__

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, fbeta_score

estrategias = {
    "Global 0.17": {
        "Muito baixa (0-10%)": 0.17,
        "Baixa (10-30%)": 0.17,
        "Moderada (30-60%)": 0.17,
        "Alta (60-100%)": 0.17
    },
    "Adaptativo conservador": {
        "Muito baixa (0-10%)": 0.15,
        "Baixa (10-30%)": 0.15,
        "Moderada (30-60%)": 0.17,
        "Alta (60-100%)": 0.17
    },
    "Adaptativo sensivel": {
        "Muito baixa (0-10%)": 0.15,
        "Baixa (10-30%)": 0.05,
        "Moderada (30-60%)": 0.17,
        "Alta (60-100%)": 0.17
    }
}

resultados_comparacao = []

for nome, mapa_limiares in estrategias.items():

    df_temp = df_validacao_faixas.copy()

    df_temp["limiar"] = (
        df_temp["faixa_historica"]
        .astype(str)
        .map(mapa_limiares)
    )

    df_temp["previsao_estrategia"] = (
        df_temp["probabilidade"] >= df_temp["limiar"]
    ).astype(int)

    for faixa, grupo in df_temp.groupby(
        "faixa_historica", observed=True
    ):

        y_real = grupo["ocorreu_sinistro"].to_numpy()
        y_pred = grupo["previsao_estrategia"].to_numpy()

        vp = int(((y_real == 1) & (y_pred == 1)).sum())
        fp = int(((y_real == 0) & (y_pred == 1)).sum())
        fn = int(((y_real == 1) & (y_pred == 0)).sum())

        resultados_comparacao.append({
            "estrategia": nome,
            "faixa": str(faixa),
            "limiar": mapa_limiares[str(faixa)],
            "precisao": precision_score(
                y_real, y_pred, zero_division=0
            ),
            "revocacao": recall_score(
                y_real, y_pred, zero_division=0
            ),
            "f2_score": fbeta_score(
                y_real, y_pred, beta=2, zero_division=0
            ),
            "vp": vp,
            "fp": fp,
            "fn": fn
        })

df_comparacao_estrategias = pd.DataFrame(
    resultados_comparacao
)

print("=" * 80)
print("SEÇÃO 9.4 — COMPARAÇÃO DETALHADA POR FAIXA")
print("=" * 80)

print("\n--- RESULTADOS POR ESTRATÉGIA E FAIXA ---")
print(
    df_comparacao_estrategias
    .round(4)
    .to_string(index=False)
)

print("\n--- FOCO: FREQUÊNCIA MUITO BAIXA (0-10%) ---")
print(
    df_comparacao_estrategias[
        df_comparacao_estrategias["faixa"]
        == "Muito baixa (0-10%)"
    ]
    .round(4)
    .to_string(index=False)
)

print("\n--- FOCO: FREQUÊNCIA BAIXA (10-30%) ---")
print(
    df_comparacao_estrategias[
        df_comparacao_estrategias["faixa"]
        == "Baixa (10-30%)"
    ]
    .round(4)
    .to_string(index=False)
)


SEÇÃO 9.4 — COMPARAÇÃO DETALHADA POR FAIXA

--- RESULTADOS POR ESTRATÉGIA E FAIXA ---
            estrategia               faixa  limiar  precisao  revocacao  f2_score    vp    fp   fn
           Global 0.17 Muito baixa (0-10%)    0.17    0.1291     0.0806    0.0872    86   580  981
           Global 0.17      Baixa (10-30%)    0.17    0.2573     0.7443    0.5399  3691 10653 1268
           Global 0.17   Moderada (30-60%)    0.17    0.4681     0.9955    0.8124  7790  8853   35
           Global 0.17      Alta (60-100%)    0.17    0.8707     1.0000    0.9711 15933  2367    0
Adaptativo conservador Muito baixa (0-10%)    0.15    0.1187     0.1200    0.1197   128   950  939
Adaptativo conservador      Baixa (10-30%)    0.15    0.2476     0.8006    0.5534  3970 12061  989
Adaptativo conservador   Moderada (30-60%)    0.17    0.4681     0.9955    0.8124  7790  8853   35
Adaptativo conservador      Alta (60-100%)    0.17    0.8707     1.0000    0.9711 15933  2367    0
   Adaptativo sensivel 